# Tutorial 2 — Point Clouds on $S^2$ and $S^3$

**MM845 — Tópicos de Geometria III: AI for Geometry**
Paired with **Lecture 2: Mathematical Foundations of Machine Learning**

---

Lecture 2 stated the thesis

$$\text{learning} \;=\; \text{approximation theory} \;+\; \text{optimisation} \;+\; \text{statistics},$$

and claimed that each ingredient carries geometric content. This notebook makes
that claim concrete on the simplest interesting manifolds available to us: the
round spheres $S^2 \subset \mathbb{R}^3$ and $S^3 \subset \mathbb{R}^4$.

Spheres are the right laboratory precisely because **we know the answers**. Every
quantity a machine-learning method will estimate — the intrinsic dimension, the
uniform measure, the geodesic distance, the eigenfunctions of the Laplacian, the
symmetry group — is available to us in closed form. When a method recovers it, we
can *prove* it did; when it fails, we can see exactly where. This is a luxury real
data never offers, and it is the reason the whole course works with geometric
datasets.

### What you will do

| § | Topic | Lecture 2 connection |
|---|---|---|
| 1 | Sampling $S^1$, $S^2$: what "uniform" means, and how to get it wrong | data generation, measure $\mu$ |
| 2 | Quasi-uniform point sets, subsampling, triangulation | representation choices |
| 3 | Chordal vs geodesic distance; the distribution of distances | metric geometry of the data |
| 4 | $SO(3)$ acting on $S^2$: group actions, invariance, orbit-aware splits | slide 11, "split by orbit, not by sample" |
| 5 | $S^3$, unit quaternions, and the Hopf fibration | visualising what you cannot draw |
| 6 | Concentration of measure and the curse of dimensionality | slide 9 |
| 7 | The manifold hypothesis: recovering $\dim = 2$ from $\mathbb{R}^{100}$ | slide 8 |
| 8 | Supervised learning on $S^2$ as energy minimisation; the U-curve | slides 4, 6, 7 |

Exercises appear throughout, marked **Exercise $n$**. They are the point of the
tutorial — the prose exists to set them up. Hints are given; the solutions are
discussed in the session.

### Before you start

You need the `aigeo` environment from **Tutorial 1**, and the kernel selected at
the top right of this window should say so. Check with the cell below; if it
fails, revisit [Tutorial 1 §9](../tutorial_01/README.md#9-troubleshooting).

In [ ]:
import sys

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers the 3d projection)
from scipy.spatial import ConvexHull, cKDTree

print("python     ", sys.version.split()[0])
print("numpy      ", np.__version__)
print("matplotlib ", plt.matplotlib.__version__)

### Reproducibility: fix the seed once, at the top

Lecture 1 listed three ingredients of a reproducible computation: code, environment,
and randomness. The third is handled here. We create one explicit random
*generator* object and pass it around, rather than calling `np.random.seed`
globally — this makes it visible in every function signature exactly which
computations are stochastic.

In [ ]:
SEED = 20260805
rng = np.random.default_rng(SEED)

# --- course palette (matches the lecture slides) --------------------------------
GEO_DARK, GEO_TEAL, GEO_RUST = "#103158", "#006c86", "#b2461e"

plt.rcParams.update({
    "figure.dpi": 110,
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.prop_cycle": plt.cycler(color=[GEO_DARK, GEO_TEAL, GEO_RUST,
                                         "#6a8caf", "#4c9a2a", "#7d3c98"]),
})

# For an interactive, rotatable 3-D view instead of static pictures:
#   pip install ipympl     and then uncomment the next line
# %matplotlib widget

Two plotting helpers we will reuse throughout. Reading them is optional; the point
of isolating them in a function, as Lecture 1 urged, is that the *interesting*
cells below then contain only mathematics.

In [ ]:
def sphere_ax(fig, pos, title="", elev=22, azim=35, lim=1.05):
    """Create a 3-D axis with equal aspect, sized for the unit sphere."""
    ax = fig.add_subplot(pos, projection="3d")
    ax.set_box_aspect((1, 1, 1))
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_zlim(-lim, lim)
    ax.view_init(elev=elev, azim=azim)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
    ax.set_title(title)
    ax.grid(False)
    return ax


def draw_wire_sphere(ax, color="0.6", alpha=0.15, n=24):
    """Faint wireframe of the unit sphere, to give the point cloud a reference."""
    u = np.linspace(0, 2 * np.pi, 2 * n)
    v = np.linspace(0, np.pi, n)
    xs = np.outer(np.cos(u), np.sin(v))
    ys = np.outer(np.sin(u), np.sin(v))
    zs = np.outer(np.ones_like(u), np.cos(v))
    ax.plot_wireframe(xs, ys, zs, color=color, alpha=alpha,
                      linewidth=0.4, rstride=2, cstride=2)
    return ax

---
## 1. Sampling: what does "uniform on $S^2$" mean?

Lecture 2 began from a measure space $(X, \mu)$ and i.i.d. samples from $\mu$. Our
first task is therefore to *produce* those samples — and the choice of $\mu$ is
already a mathematical decision, not a technicality.

For a sphere the canonical choice is the unique probability measure invariant
under the isometry group $O(n)$; equivalently, normalised Riemannian volume. Any
sampling scheme must be checked against that definition, because the obvious
scheme is wrong.

### 1.1 The naive scheme, and why it fails

Parametrise $S^2$ by spherical coordinates,

$$\Phi(\theta,\varphi) = (\sin\varphi\cos\theta,\; \sin\varphi\sin\theta,\; \cos\varphi),
\qquad \theta \in [0,2\pi),\; \varphi \in [0,\pi].$$

It is tempting to draw $\theta$ and $\varphi$ uniformly from their ranges. But the
pullback of the area form is

$$\Phi^*(\mathrm{d}A) = \sin\varphi \, \mathrm{d}\varphi \wedge \mathrm{d}\theta \;\neq\; \mathrm{d}\varphi \wedge \mathrm{d}\theta ,$$

so uniform $(\theta,\varphi)$ over-weights the poles, where the Jacobian factor
$\sin\varphi$ vanishes. This is the same fact that makes a Mercator map of the
Earth exaggerate Greenland.

### 1.2 Two schemes that work

**Gaussian normalisation (Muller's method).** If $x \sim \mathcal{N}(0, I_n)$ then
its density $\propto e^{-\|x\|^2/2}$ is rotation-invariant, hence so is the law of
$x/\|x\|$. This works in *every* dimension — we will use it for $S^3$ too.

**Archimedes' theorem.** The projection $S^2 \to [-1,1]$, $(x,y,z) \mapsto z$, is
measure-preserving up to the constant $2\pi$: the area of a spherical zone depends
only on its height. So drawing $z \sim U[-1,1]$ and $\theta \sim U[0,2\pi)$ gives
a uniform point. This is special to $S^2$.

In [ ]:
def sample_sphere(n_points, dim, rng):
    """n_points uniform samples on S^{dim-1} = unit sphere in R^dim (Muller)."""
    x = rng.normal(size=(n_points, dim))
    return x / np.linalg.norm(x, axis=1, keepdims=True)


def sample_naive_angles(n_points, rng):
    """WRONG for S^2: uniform in the coordinate rectangle, not in area."""
    theta = rng.uniform(0, 2 * np.pi, n_points)   # azimuth
    phi = rng.uniform(0, np.pi, n_points)         # polar angle
    return np.stack([np.sin(phi) * np.cos(theta),
                     np.sin(phi) * np.sin(theta),
                     np.cos(phi)], axis=1)


def sample_archimedes(n_points, rng):
    """Correct for S^2: uniform height z, uniform azimuth."""
    z = rng.uniform(-1, 1, n_points)
    theta = rng.uniform(0, 2 * np.pi, n_points)
    r = np.sqrt(1 - z**2)
    return np.stack([r * np.cos(theta), r * np.sin(theta), z], axis=1)


N = 2000
X_gauss = sample_sphere(N, 3, rng)
X_naive = sample_naive_angles(N, rng)
X_arch = sample_archimedes(N, rng)

# Sanity check first, plot second: every point must have unit norm.
for name, X in [("gaussian", X_gauss), ("naive", X_naive), ("archimedes", X_arch)]:
    assert np.allclose(np.linalg.norm(X, axis=1), 1.0), name
print("all three samples lie on the unit sphere")

Now look at them. The bias is visible without any statistics — the naive cloud has
conspicuous tufts at the poles.

In [ ]:
fig = plt.figure(figsize=(10.5, 3.8))
for k, (name, X) in enumerate([("naive $(\\theta,\\varphi)$ — biased", X_naive),
                               ("Gaussian normalisation", X_gauss),
                               ("Archimedes ($z$ uniform)", X_arch)]):
    ax = sphere_ax(fig, 131 + k, name, elev=12)
    draw_wire_sphere(ax)
    ax.scatter(X[:, 0], X[:, 1], X[:, 2], s=3, c=GEO_TEAL, alpha=0.55, lw=0)
fig.suptitle("2000 points on $S^2$, three sampling schemes", y=1.02)
plt.tight_layout(); plt.show()

### 1.3 Detecting the bias quantitatively

Eyeballing a picture is not a test. Archimedes' theorem gives us one: for a
*uniform* point on $S^2$, the coordinate $z$ is exactly uniform on $[-1,1]$. So
histogram $z$ and compare against the flat density $\tfrac12$.

This is the pattern to internalise, and it recurs for the rest of the course:
**find a scalar statistic whose exact distribution you know, and check it.**

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10.5, 2.9), sharey=True)
for ax, (name, X) in zip(axes, [("naive", X_naive),
                                ("Gaussian", X_gauss),
                                ("Archimedes", X_arch)]):
    ax.hist(X[:, 2], bins=30, range=(-1, 1), density=True,
            color=GEO_TEAL, alpha=0.75, edgecolor="white", lw=0.4)
    ax.axhline(0.5, color=GEO_RUST, lw=1.6, label=r"truth: $\rho(z)=\frac{1}{2}$")
    ax.set_title(name); ax.set_xlabel("$z$")
axes[0].set_ylabel("density"); axes[0].legend(fontsize=7)
fig.suptitle("Distribution of the height coordinate (Archimedes' theorem)", y=1.04)
plt.tight_layout(); plt.show()

> **Exercise 1 — a second, independent test.**
> The $z$-histogram checks the marginal in one direction only; a scheme could pass
> it and still be non-uniform in $\theta$.
>
> (a) Choose a fixed unit vector $u$ and histogram $\langle X_i, u \rangle$. What
> must the answer be, and why does taking a *random* $u$ make the test stronger?
>
> (b) Estimate the area of the spherical cap $\{z > 0.8\}$ by the fraction of
> sample points inside it, for each of the three schemes. Compare with the exact
> value $\tfrac{1-0.8}{2} = 0.1$ and report the Monte-Carlo error $\sim 1/\sqrt N$.
>
> (c) The naive scheme is *correct* for $S^1$ with uniform $\theta$. Why? At which
> step of §1.1 does the argument for $S^2$ break down, and what happens on $S^{n-1}$
> for $n \geq 4$?


**Minha Resposta:**

(a) Como a distribuição é invariante por rotação, $u$ pode ser tomado como $z$. Dessa forma, $\langle X_i, u \rangle = z(X_i)$ que, como dito, tem distribuição uniforme em $[-1,1]$. Tomar $u$ aleatório impede de alguma simetria mascarar, coincidentemente, a distribuição na direção z.

In [ ]:
u = rng.normal(size=(1, 3))
u = u / np.linalg.norm(u)

fig, axes = plt.subplots(1, 3, figsize=(10.5, 2.9), sharey=True)
for ax, (name, X) in zip(axes, [("naive", X_naive),
                                ("Gaussian", X_gauss),
                                ("Archimedes", X_arch)]):
    ax.hist(np.inner(X,u), bins=30, range=(-1, 1), density=True,
            color=GEO_TEAL, alpha=0.75, edgecolor="white", lw=0.4)
    ax.axhline(0.5, color=GEO_RUST, lw=1.6, label=r"truth: $\rho(z)=\frac{1}{2}$")
    ax.set_title(name); ax.set_xlabel(r"$\langle x, u \rangle$")
axes[0].set_ylabel("density"); axes[0].legend(fontsize=7)
fig.suptitle("Distribution of the inner product with $u$ coordinate (Archimedes' theorem)", y=1.04)
plt.tight_layout(); plt.show()

(b)

In [ ]:
def cap_fraction(X, height):
    """Fraction of the sample lying in the cap {z > height}."""
    return np.mean(X[:, 2] > height)

mc_error = np.sqrt(0.1 * 0.9 / N)  # onde N = 2000
print("Monte-Carlo error:", f"{mc_error:.2f}", "\n")

print(f"{'Model':<12} | {'Value':<8} | {'Error':<8} | {'Inside'}")
print("-------------------------------------------")

for ax, (name, X) in zip(axes, [("Naive", X_naive),
                                ("Gaussian", X_gauss),
                                ("Archimedes", X_arch)]):
    cf = cap_fraction(X, 0.8)
    print(f"{name+":":<12} | {cf:<8} | {abs(cf-0.1):<8.3f} | {abs(cf-0.1) < mc_error}")

(c) Porque a parametrização de $S^1$ preserva volume, enquanto de $S^n$, $n \geq 2$, não.

> **Exercise 2 — $S^1$, and the failure of the obvious generalisation.**
> Write `sample_circle_naive(n, rng)` using uniform $\theta$, and verify it is
> uniform. Then attempt the direct analogue of Archimedes on $S^3$: draw the last
> coordinate uniformly on $[-1,1]$ and the remaining three uniformly on a sphere of
> the matching radius. Test the result against `sample_sphere(n, 4, rng)` using the
> statistic from Exercise 1(a). What is the true density of a single coordinate of a
> uniform point on $S^{n-1}$? (You will derive it in §6 — attempt it first.)

**Minha Resposta:**

$S^1$:

In [ ]:
def sample_circle_naive(n_points, rng):
    theta = rng.uniform(0, 2 * np.pi, n_points) 
    return np.stack([np.cos(theta), np.sin(theta)], axis=1)

X_circle = sample_circle_naive(N, rng)

fig, ax = plt.subplots(figsize=(5.6, 3.0))
ax.hist(np.arctan2(X_circle[:,1], X_circle[:,0]), density=True, alpha=0.6, color=GEO_TEAL)
ax.axhline(1/(2*np.pi), color=GEO_RUST, lw=1.6, label=r"truth: $\rho(z)=\frac{1}{2\pi}$")
ax.set_xlabel(r"$\theta$"); ax.set_ylabel("density")
ax.set_title(f"Covering quality, $N={N}$"); ax.legend()
plt.tight_layout(); plt.show()

$S^3$:

In [ ]:
def sample_archimedes_S3(n_points, rng):
    w = rng.uniform(-1, 1, n_points)
    r = np.sqrt(1 - w**2)
    S2 = sample_archimedes(n_points, rng)
    return np.stack([r * S2[:,0], r * S2[:,1], r * S2[:,2], w], axis=1)

X_arch_S3 = sample_archimedes_S3(N, rng)
X_gauss_S3 = sample_sphere(N, 4, rng)

u = rng.normal(size=(1, 4))
u = u / np.linalg.norm(u)

fig, axes = plt.subplots(1, 2, figsize=(10.5, 2.9), sharey=True)
for ax, (name, X) in zip(axes, [("Gaussian", X_gauss_S3),
                                ("Archimedes", X_arch_S3)]):
    ax.hist(np.inner(X,u), bins=30, range=(-1, 1), density=True,
            color=GEO_TEAL, alpha=0.75, edgecolor="white", lw=0.4)
    ax.axhline(0.5, color=GEO_RUST, lw=1.6, label=r"uniform: $\rho(z)=\frac{1}{2}$")
    ax.set_title(name); ax.set_xlabel(r"$\langle x, u \rangle$")
axes[0].set_ylabel("density"); axes[0].legend(fontsize=7)
fig.suptitle("Distribution of the inner product with $u$ coordinate (Archimedes' theorem)", y=1.04)
plt.tight_layout(); plt.show()


---
## 2. Structured point sets: quasi-uniform, subsampled, triangulated

Random samples are the right model for *data*, but they are a poor way to *cover* a
manifold: independent points clump, leaving holes of size $\sim \sqrt{\log N / N}$.
When the point cloud is a numerical discretisation rather than a dataset — a
quadrature rule, a set of collocation points for a PINN in Lecture 12 — we want
something more even.

### 2.1 The Fibonacci lattice

Place points at heights $z_i = 1 - \frac{2i+1}{N}$ (equally spaced, by Archimedes)
and advance the azimuth by the golden angle $2\pi/\phi$, $\phi = \frac{1+\sqrt5}{2}$,
at each step. Because $\phi$ is the irrational number worst approximable by
rationals, the azimuths never come close to repeating, and the points spiral into a
remarkably even cover.

In [ ]:
GOLDEN = (1 + 5**0.5) / 2


def fibonacci_sphere(n_points):
    """Deterministic quasi-uniform point set on S^2."""
    i = np.arange(n_points)
    z = 1 - 2 * (i + 0.5) / n_points
    r = np.sqrt(np.maximum(0.0, 1 - z**2))
    theta = 2 * np.pi * i / GOLDEN
    return np.stack([r * np.cos(theta), r * np.sin(theta), z], axis=1)


X_fib = fibonacci_sphere(N)

fig = plt.figure(figsize=(7.5, 3.8))
for k, (name, X) in enumerate([("i.i.d. uniform", X_gauss), ("Fibonacci lattice", X_fib)]):
    ax = sphere_ax(fig, 121 + k, name, elev=20)
    ax.scatter(X[:, 0], X[:, 1], X[:, 2], s=3.5, c=GEO_DARK, alpha=0.7, lw=0)
plt.tight_layout(); plt.show()

The difference is quantitative, not merely aesthetic. Compare the distribution of
nearest-neighbour distances: the random cloud has both very close pairs (clumps)
and isolated points, while the lattice is sharply concentrated.

In [ ]:
def nn_distances(X):
    """Distance from each point to its nearest *other* point."""
    tree = cKDTree(X)
    d, _ = tree.query(X, k=2)   # column 0 is the point itself, at distance 0
    return d[:, 1]


d_rand, d_fib = nn_distances(X_gauss), nn_distances(X_fib)

fig, ax = plt.subplots(figsize=(5.6, 3.0))
bins = np.linspace(0, max(d_rand.max(), d_fib.max()) * 1.02, 45)
ax.hist(d_rand, bins=bins, density=True, alpha=0.6, color=GEO_TEAL, label="i.i.d. uniform")
ax.hist(d_fib, bins=bins, density=True, alpha=0.6, color=GEO_RUST, label="Fibonacci")
ax.set_xlabel("nearest-neighbour distance"); ax.set_ylabel("density")
ax.set_title(f"Covering quality, $N={N}$"); ax.legend()
plt.tight_layout(); plt.show()

for name, d in [("i.i.d. uniform", d_rand), ("Fibonacci   ", d_fib)]:
    print(f"{name}: mean {d.mean():.4f}   min {d.min():.5f}   "
          f"max/min {d.max()/d.min():7.1f}")

### 2.2 Farthest point sampling

A common manipulation: given a large cloud, extract a small, well-spread subset.
Greedy *farthest point sampling* repeatedly adds the point maximising the distance
to the set chosen so far. It is $O(kN)$ and gives a $2$-approximation to the
optimal covering radius.

In [ ]:
def farthest_point_sample(X, k, start=0):
    """Greedy k-subset of X maximising minimum pairwise distance (approximately)."""
    idx = np.empty(k, dtype=int)
    idx[0] = start
    d = np.linalg.norm(X - X[start], axis=1)
    for i in range(1, k):
        idx[i] = np.argmax(d)
        d = np.minimum(d, np.linalg.norm(X - X[idx[i]], axis=1))
    return idx


sub = farthest_point_sample(X_gauss, 64)

fig = plt.figure(figsize=(7.5, 3.8))
ax = sphere_ax(fig, 121, "cloud + farthest-point subset", elev=20)
ax.scatter(*X_gauss.T, s=2, c="0.75", alpha=0.5, lw=0)
ax.scatter(*X_gauss[sub].T, s=26, c=GEO_RUST, depthshade=False)

ax = sphere_ax(fig, 122, "random subset of the same size", elev=20)
rand_sub = rng.choice(len(X_gauss), 64, replace=False)
ax.scatter(*X_gauss.T, s=2, c="0.75", alpha=0.5, lw=0)
ax.scatter(*X_gauss[rand_sub].T, s=26, c=GEO_TEAL, depthshade=False)
plt.tight_layout(); plt.show()

### 2.3 From point cloud to triangulated surface

A point cloud on a *convex* surface carries a canonical triangulation: the boundary
of its convex hull. For points on $S^2$ this is the spherical Delaunay
triangulation, and it is the bridge from "cloud" to the simplicial complexes used
in Tutorials 6, 9 and 11.

In [ ]:
X_small = fibonacci_sphere(300)
hull = ConvexHull(X_small)
print(f"{len(X_small)} vertices, {len(hull.simplices)} triangles, "
      f"{len(hull.simplices) * 3 // 2} edges")
print("Euler characteristic V - E + F =",
      len(X_small) - len(hull.simplices) * 3 // 2 + len(hull.simplices))

fig = plt.figure(figsize=(4.6, 4.4))
ax = sphere_ax(fig, 111, "convex hull of a Fibonacci cloud on $S^2$", elev=18)
ax.plot_trisurf(X_small[:, 0], X_small[:, 1], X_small[:, 2],
                triangles=hull.simplices, color=GEO_TEAL, alpha=0.55,
                edgecolor="white", linewidth=0.25)
plt.tight_layout(); plt.show()

> **Exercise 3 — discretisation quality.**
> (a) The Euler characteristic printed above should be $2$. Confirm it, and explain
> why it is a genuine test of the triangulation rather than a tautology.
>
> (b) Use the triangulation to estimate the surface area of $S^2$ by summing
> triangle areas. Plot the relative error against $N$ on log-log axes for
> $N = 50, 100, \dots, 5000$, for both the Fibonacci and the random cloud. What
> convergence rates do you observe, and which one would you want for a quadrature
> rule?
>
> (c) *(harder)* The Fibonacci lattice is not exactly uniform near the poles. Detect
> this by plotting the triangle-area distribution as a function of $|z|$.

**Minha Resposta:**

(a) A triangulação pode não detectar buracos e dar uma relação distinta.

(b) 

In [ ]:
def triangle_areas(X, triangles):
    """Área de cada triângulo usando produto vetorial."""
    v0 = X[triangles[:, 0]]
    v1 = X[triangles[:, 1]]
    v2 = X[triangles[:, 2]]
    
    # Produto vetorial (v1-v0) × (v2-v0)
    cross = np.cross(v1 - v0, v2 - v0)
    areas = 0.5 * np.linalg.norm(cross, axis=1)
    return areas

# Varrer N
Ns = np.array([50, 100, 150, 200, 300, 500, 1000, 2000, 5000])
true_area = 4 * np.pi  # área de S^2
errors_fib = []
errors_rand = []

for N_test in Ns:
    # Fibonacci
    X_fib_test = fibonacci_sphere(N_test)
    hull_fib = ConvexHull(X_fib_test)
    area_fib = triangle_areas(X_fib_test, hull_fib.simplices).sum()
    errors_fib.append(np.abs(area_fib - true_area) / true_area)
    
    # Random
    X_rand_test = sample_sphere(N_test, 3, rng)
    hull_rand = ConvexHull(X_rand_test)
    area_rand = triangle_areas(X_rand_test, hull_rand.simplices).sum()
    errors_rand.append(np.abs(area_rand - true_area) / true_area)

# Plot log-log
fig, ax = plt.subplots(figsize=(7.0, 4.0))
ax.loglog(Ns, errors_fib, "o-", color=GEO_RUST, lw=2, label="Fibonacci", ms=6)
ax.loglog(Ns, errors_rand, "s-", color=GEO_TEAL, lw=2, label="Random", ms=6)

# Regressão para taxas de convergência
from numpy.polynomial import Polynomial
log_N = np.log(Ns)
log_err_fib = np.log(errors_fib)
log_err_rand = np.log(errors_rand)

# Ajustar log(err) = a + b*log(N)  ->  err ~ N^b
p_fib = Polynomial.fit(log_N, log_err_fib, 1)
p_rand = Polynomial.fit(log_N, log_err_rand, 1)
rate_fib = p_fib.convert().coef[1]
rate_rand = p_rand.convert().coef[1]

ax.loglog(Ns, errors_fib[0] * (Ns / Ns[0])**rate_fib, "--", 
          color=GEO_RUST, alpha=0.5, label=f"Fib: $N^{{{rate_fib:.2f}}}$")
ax.loglog(Ns, errors_rand[0] * (Ns / Ns[0])**rate_rand, "--",
          color=GEO_TEAL, alpha=0.5, label=f"Rand: $N^{{{rate_rand:.2f}}}$")

ax.set_xlabel("$N$ (number of points)"); ax.set_ylabel("relative error")
ax.set_title("Surface area estimation: convergence rates")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f"Fibonacci convergence rate: {rate_fib:.3f} (should be ~ -2/3 for quadrature)")
print(f"Random convergence rate:    {rate_rand:.3f}")

(c)

In [ ]:
# Triangulação da Fibonacci
X_fib_test = fibonacci_sphere(1000)
hull_fib_test = ConvexHull(X_fib_test)

# Calcular áreas e altura |z| de cada triângulo
areas = triangle_areas(X_fib_test, hull_fib_test.simplices)
z_centers = X_fib_test[hull_fib_test.simplices].mean(axis=1)[:, 2]  # centro de cada triângulo

fig, ax = plt.subplots(figsize=(7.0, 4.0))

# Scatter plot: |z| vs área
ax.scatter(np.abs(z_centers), areas, s=2, c=GEO_TEAL, alpha=0.5)

# Valor esperado: área média (uniforme)
expected_area = 4 * np.pi / len(areas)
ax.axhline(expected_area, color=GEO_RUST, lw=2, ls="--", 
           label=f"Expected area (uniform): {expected_area:.5f}")

ax.set_xlabel("$|z|$ (distance from equator)")
ax.set_ylabel("Triangle area")
ax.set_title("Fibonacci lattice: non-uniformity near poles")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# Estatísticas por zona
zones = [(0, 0.2), (0.2, 0.4), (0.4, 0.6), (0.6, 0.8), (0.8, 1.0)]
for z_min, z_max in zones:
    mask = (np.abs(z_centers) >= z_min) & (np.abs(z_centers) < z_max)
    mean_area = areas[mask].mean()
    print(f"|z| ∈ [{z_min:.1f}, {z_max:.1f}): mean area = {mean_area:.6f} "
          f"(ratio to expected: {mean_area/expected_area:.3f})")

In [ ]:
def gram_distances(X):
    """Pairwise geodesic and chordal distance matrices, fully vectorised."""
    C = np.clip(X @ X.T, -1.0, 1.0)      # clip: guards arccos against round-off
    geo = np.arccos(C)
    chord = np.sqrt(np.maximum(0.0, 2 - 2 * C))
    return geo, chord


Xs = X_gauss[:600]
geo, chord = gram_distances(Xs)
iu = np.triu_indices(len(Xs), k=1)       # upper triangle: each pair once
print("geodesic  range", geo[iu].min().round(4), geo[iu].max().round(4))
print("chordal   range", chord[iu].min().round(4), chord[iu].max().round(4))
print("identity  2*sin(d_geo/2) == d_chord :",
      np.allclose(2 * np.sin(geo / 2), chord, atol=1e-8))

### The distribution of pairwise distances is known exactly

For two *independent* uniform points on $S^2$, Archimedes again: fix the first
point, rotate it to the north pole, and the inner product becomes the height
coordinate of the second — so

$$\langle X, Y \rangle \sim U[-1,1],
\qquad\text{hence}\qquad
\rho_{\text{geo}}(d) = \tfrac12 \sin d \;\text{ on } [0,\pi].$$

A one-line theoretical prediction we can overlay on the data. Note that this is
*not* a property of the sample; it is a property of $\mu$, and the histogram
converging to it is the law of large numbers from Lecture 2, slide 6.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.0, 3.0))

axes[0].hist(np.cos(geo[iu]), bins=40, density=True, color=GEO_TEAL,
             alpha=0.75, edgecolor="white", lw=0.4)
axes[0].axhline(0.5, color=GEO_RUST, lw=1.6, label=r"$U[-1,1]$")
axes[0].set_xlabel(r"$\langle x,y\rangle$"); axes[0].set_title("inner products")
axes[0].legend(fontsize=8)

t = np.linspace(0, np.pi, 200)
axes[1].hist(geo[iu], bins=40, density=True, color=GEO_TEAL,
             alpha=0.75, edgecolor="white", lw=0.4)
axes[1].plot(t, 0.5 * np.sin(t), color=GEO_RUST, lw=1.6,
             label=r"$\frac{1}{2}\sin d$")
axes[1].set_xlabel("geodesic distance"); axes[1].set_title("geodesic distances")
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

### Geodesics, drawn

The geodesic from $p$ to $q$ is the great-circle arc, given by *spherical linear
interpolation*:

$$\gamma(t) = \frac{\sin\big((1-t)\omega\big)\,p + \sin(t\omega)\,q}{\sin\omega},
\qquad \omega = \arccos\langle p,q\rangle .$$

Check for yourself that $\|\gamma(t)\| = 1$ and $\gamma(0)=p$, $\gamma(1)=q$.

In [ ]:
def slerp(p, q, t):
    """Great-circle arc from p to q, sampled at parameters t in [0,1]."""
    t = np.atleast_1d(t)
    omega = np.arccos(np.clip(p @ q, -1.0, 1.0))
    if omega < 1e-9:
        return np.tile(p, (len(t), 1))
    return (np.sin((1 - t)[:, None] * omega) * p
            + np.sin(t[:, None] * omega) * q) / np.sin(omega)


anchors = sample_sphere(6, 3, rng)
tt = np.linspace(0, 1, 120)

fig = plt.figure(figsize=(4.8, 4.6))
ax = sphere_ax(fig, 111, "geodesic arcs between random points on $S^2$", elev=18)
draw_wire_sphere(ax, alpha=0.10)
for i in range(len(anchors)):
    for j in range(i + 1, len(anchors)):
        arc = slerp(anchors[i], anchors[j], tt)
        ax.plot(*arc.T, color=GEO_TEAL, lw=1.1, alpha=0.8)
ax.scatter(*anchors.T, s=45, c=GEO_RUST, depthshade=False, zorder=5)
plt.tight_layout(); plt.show()

# the arc really is on the sphere, and really is shorter than any polyline in R^3
arc = slerp(anchors[0], anchors[1], tt)
assert np.allclose(np.linalg.norm(arc, axis=1), 1.0)
length = np.linalg.norm(np.diff(arc, axis=0), axis=1).sum()
print(f"polygonal length {length:.5f}   vs   arccos<p,q> = "
      f"{np.arccos(anchors[0] @ anchors[1]):.5f}")

> **Exercise 4 — the metric matters.**
> (a) Repeat the distance-distribution plot for $S^{n-1}$ with $n = 2, 3, 10$. For
> $S^1$ the geodesic distance is uniform on $[0,\pi]$; for $S^2$ it is
> $\frac12 \sin d$. Guess and then verify the general density
> $\rho_n(d) \propto \sin^{n-2} d$. Where does the exponent come from geometrically?
>
> (b) Build the $k$-nearest-neighbour graph of a cloud on $S^2$ under each metric
> and confirm they coincide. Construct a manifold and a sampling for which the
> nearest-neighbour graphs would *not* coincide — what property of the sphere are
> you using?
>
> (c) Compute the *graph* distance in the $k$-NN graph (shortest path, with edges
> weighted by chordal length) and plot it against the true geodesic distance. For
> which $k$ is the approximation good? This is the core idea of Isomap, and a
> preview of Tutorial 8.

**Minha Resposta:**

**(a)** Para dois pontos uniformes em $S^{n-1}\subset\mathbb{R}^n$, por invariância rotacional podemos fixar o primeiro ponto no polo norte. Se $D$ é a distância geodésica até o segundo ponto, então

$$Z=\langle X,Y\rangle=\cos D.$$

A esfera geodésica de raio $d$ em $S^{n-1}$ é uma cópia de $S^{n-2}$ com raio extrínseco $\sin d$. Portanto, seu volume é

$$\operatorname{Vol}(S^{n-2})\sin^{n-2}(d).$$

Dividindo pelo volume total de $S^{n-1}$, obtemos a densidade da distância geodésica:

$$f_D(d)=\frac{\operatorname{Vol}(S^{n-2})}{\operatorname{Vol}(S^{n-1})}\sin^{n-2}(d),\qquad 0\le d\le\pi.$$

O expoente $n-2$ vem geometricamente da dimensão da esfera geodésica: ao aumentar seu raio, cada uma das suas $n-2$ direções escala por $\sin d$. Em particular, para $S^1$, $f_D(d)=1/\pi$; para $S^2$, $f_D(d)=\frac12\sin d$.

In [ ]:
import math

from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import shortest_path


def sphere_volume(m):
    """Volume of the unit sphere S^m."""
    return 2 * np.pi ** ((m + 1) / 2) / math.gamma((m + 1) / 2)


# (a) Distâncias de um ponto fixo a amostras uniformes em S^{n-1}.
# Em R^n, o polo norte é o vetor e_n; por isso usamos a última coordenada.
dims_ex4 = [2, 3, 10]
t = np.linspace(0, np.pi, 400)
fig, axes = plt.subplots(1, 3, figsize=(10.5, 2.9), sharey=False)

for ax, n in zip(axes, dims_ex4):
    Xn = sample_sphere(N, n, rng)
    d = np.arccos(np.clip(Xn[:, -1], -1.0, 1.0))
    constant = sphere_volume(n - 2) / sphere_volume(n - 1)
    density = constant * np.sin(t) ** (n - 2)
    ax.hist(d, bins=50, range=(0, np.pi), density=True,
            color=GEO_TEAL, alpha=0.72, edgecolor="white", lw=0.3)
    ax.plot(t, density, color="red", lw=1.8,
            label=r"teoria: $f_D(d)=C_n\sin^{n-2}(d)$")
    ax.set_title(rf"$S^{{{n-1}}}$ (n={n})")
    ax.set_xlabel("distância geodésica $d$")
    ax.legend(fontsize=7)
axes[0].set_ylabel("densidade")
fig.suptitle("Distância geodésica até o polo norte", y=1.04)
plt.tight_layout(); plt.show()

**(b)** No intervalo $[0,\pi]$ temos

$$d_{\mathrm{ch}}=\sqrt{2-2\cos d}=2\sin(d/2).$$

A função $d\mapsto 2\sin(d/2)$ é estritamente crescente. Logo, para cada ponto, ordenar os outros pontos por distância geodésica ou por distância chordal produz exatamente os mesmos vizinhos. Assim, os grafos k-NN coincidem em $S^2$.

Isso usa uma propriedade especial da esfera: a distância chordal depende apenas da distância geodésica e por uma função crescente. Em uma variedade geral, ou com uma amostragem em que a distância ambiente não preserve essa ordenação, os dois grafos podem ser diferentes, como é o caso para $[0,2\pi] \sim S^1 \backslash \{ 1 \} \subseteq \mathbb{C}$.

In [ ]:
# (b) Os k vizinhos coincidem: as duas métricas são funções crescentes uma da outra.
X_knn = sample_sphere(350, 3, rng)
G = np.clip(X_knn @ X_knn.T, -1.0, 1.0)
D_geo = np.arccos(G)
D_ch = np.sqrt(np.maximum(0.0, 2.0 - 2.0 * G))
np.fill_diagonal(D_geo, np.inf)
np.fill_diagonal(D_ch, np.inf)

for k in [3, 8, 16]:
    geo_neighbours = np.argsort(D_geo, axis=1)[:, :k]
    chord_neighbours = np.argsort(D_ch, axis=1)[:, :k]
    print(f"k={k}: grafos coincidem =", np.array_equal(geo_neighbours, chord_neighbours))

**(c)** No grafo k-NN com pesos chordais, a distância entre dois pontos é o menor caminho ponderado:

$$d_G(i,j)=\min_{i=v_0\to\cdots\to v_m=j}\sum_{r=0}^{m-1}d_{\mathrm{ch}}(v_r,v_{r+1}).$$

Para $k$ pequeno, o grafo pode ficar desconectado ou forçar caminhos muito tortuosos. À medida que $k$ cresce, ele fica conectado e $d_G$ aproxima a distância geodésica. Para $k$ demasiado grande, os caminhos podem usar atalhos através do interior da bola chordal e a aproximação piora. Portanto, espera-se um intervalo intermediário de bons valores de $k$.

In [ ]:
# (c) Distâncias no grafo k-NN, com pesos chordais, contra a geodésica verdadeira.
def knn_graph_distance(X, k):
    gram = np.clip(X @ X.T, -1.0, 1.0)
    weights = np.sqrt(np.maximum(0.0, 2.0 - 2.0 * gram))
    np.fill_diagonal(weights, np.inf)
    neighbours = np.argsort(weights, axis=1)[:, :k]
    rows = np.repeat(np.arange(len(X)), k)
    cols = neighbours.ravel()
    data = weights[rows, cols]
    # Tornar o grafo não-direcionado para permitir caminhos nos dois sentidos.
    graph = csr_matrix((data, (rows, cols)), shape=(len(X), len(X)))
    graph = graph.minimum(graph.T)
    return shortest_path(graph, directed=False)

X_graph = sample_sphere(250, 3, rng)
D_true = np.arccos(np.clip(X_graph @ X_graph.T, -1.0, 1.0))
upper = np.triu_indices_from(D_true, k=1)
fig, axes = plt.subplots(1, 3, figsize=(10.5, 2.9), sharex=True, sharey=True)
for ax, k in zip(axes, [4, 8, 16]):
    D_graph = knn_graph_distance(X_graph, k)
    valid = np.isfinite(D_graph[upper])
    ax.scatter(D_true[upper][valid], D_graph[upper][valid],
               s=3, alpha=0.25, color=GEO_TEAL)
    lim = np.pi
    ax.plot([0, lim], [0, lim], "--", color=GEO_RUST, lw=1.2)
    rel_error = np.mean(np.abs(D_graph[upper][valid] - D_true[upper][valid]) /
                        np.maximum(D_true[upper][valid], 1e-12))
    ax.set_title(rf"$k={k}$, erro médio = {rel_error:.3f}")
    ax.set_xlabel("true $d_{geo}$")
axes[0].set_ylabel("graph distance $d_G$")
fig.suptitle("Isomap: distância no grafo versus distância geodésica", y=1.04)
plt.tight_layout(); plt.show()

---
## 4. Group actions: $SO(3)$ on $S^2$

Every statement so far was invariant under rotation, because $\mu$ is. Symmetry is
the geometric prior par excellence, and Lecture 10 will build it into the
hypothesis space $\mathcal{H}$ itself. Here we set up the machinery and draw the
practical consequence Lecture 2 flagged on slide 11: **when your data has a
symmetry, a random train/test split leaks information.**

We generate rotations from unit quaternions — which is to say, from a uniform
sample on $S^3$. The double cover $S^3 \to SO(3)$ sends $q$ and $-q$ to the same
rotation, and pushes the uniform measure on $S^3$ forward to the Haar measure on
$SO(3)$; so "uniformly random rotation" is a $S^3$ sampling problem, and we already
solved it.

In [ ]:
def quat_to_rotation(q):
    """Unit quaternion q = (w, x, y, z)  ->  rotation matrix in SO(3)."""
    w, x, y, z = q
    return np.array([
        [1 - 2 * (y * y + z * z), 2 * (x * y - w * z),     2 * (x * z + w * y)],
        [2 * (x * y + w * z),     1 - 2 * (x * x + z * z), 2 * (y * z - w * x)],
        [2 * (x * z - w * y),     2 * (y * z + w * x),     1 - 2 * (x * x + y * y)],
    ])


def random_rotation(rng):
    """Haar-uniform element of SO(3), via a uniform point of S^3."""
    return quat_to_rotation(sample_sphere(1, 4, rng)[0])


R = random_rotation(rng)
print("R R^T = I :", np.allclose(R @ R.T, np.eye(3)))
print("det R     :", np.linalg.det(R).round(12))
print("q and -q give the same rotation :",
      np.allclose(quat_to_rotation(np.array([0.5, 0.5, 0.5, 0.5])),
                  quat_to_rotation(-np.array([0.5, 0.5, 0.5, 0.5]))))

### Invariance, tested

Two things must hold, and both are worth checking numerically before trusting any
downstream experiment: the *sample distribution* is rotation-invariant, and the
*metric* is rotation-invariant.

Watch the third line of output carefully — it is a lesson in numerical analysis, not
a bug in the mathematics.

In [ ]:
X_rot = X_gauss @ R.T                       # rotate every point (row-vector convention)

G_before = X_gauss[:400] @ X_gauss[:400].T
G_after = X_rot[:400] @ X_rot[:400].T
print("inner products preserved :", np.allclose(G_before, G_after, atol=1e-12))

geo_before, ch_before = gram_distances(X_gauss[:400])
geo_after, ch_after = gram_distances(X_rot[:400])
print("chordal distances        :", np.allclose(ch_before, ch_after, atol=1e-10))
print("geodesic distances       :", np.allclose(geo_before, geo_after, atol=1e-10),
      "  max discrepancy", f"{np.abs(geo_before - geo_after).max():.2e}")

fig, ax = plt.subplots(figsize=(5.4, 2.9))
ax.hist(X_gauss[:, 2], bins=30, range=(-1, 1), density=True, alpha=0.6,
        color=GEO_TEAL, label="original $z$")
ax.hist(X_rot[:, 2], bins=30, range=(-1, 1), density=True, alpha=0.6,
        color=GEO_RUST, label="rotated $z$")
ax.axhline(0.5, color="k", lw=1.2, ls="--")
ax.set_title("the law of the sample is rotation-invariant")
ax.set_xlabel("$z$"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

**Why the geodesic check fails at `atol=1e-10`.** Inner products and chordal
distances are preserved to machine precision, but $\arccos$ is *ill-conditioned*
near $\pm 1$: its derivative $-1/\sqrt{1-t^2}$ blows up there. A perturbation of
size $\epsilon$ in $\langle x,y\rangle$ becomes one of size $\sqrt{\epsilon}$ in the
geodesic distance, so nearly-coincident points lose about half their significant
digits — including the diagonal, where the true distance is $0$.

The moral is general and worth carrying into every later tutorial: **an identity
that holds exactly in the mathematics need not hold to `atol=1e-10` in floating
point**, and the right response is to test the well-conditioned quantity. Prefer
storing $\langle x, y\rangle$ and converting only when you need an angle.

### Orbits

Fix one rotation $R$ and iterate it on a point: the orbit of the cyclic group
$\langle R \rangle$ traces a circle of latitude about the rotation axis. Fix
instead a *point* and act by many random rotations: the orbit is the whole sphere.
The picture below shows both, plus the orbit of a small cap.

In [ ]:
def rotation_axis_angle(axis, angle):
    """Rodrigues' formula: rotation by `angle` about a unit `axis`."""
    axis = axis / np.linalg.norm(axis)
    K = np.array([[0, -axis[2], axis[1]],
                  [axis[2], 0, -axis[0]],
                  [-axis[1], axis[0], 0]])
    return np.eye(3) + np.sin(angle) * K + (1 - np.cos(angle)) * (K @ K)


axis = np.array([0.3, 0.5, 0.81])
axis /= np.linalg.norm(axis)
R_small = rotation_axis_angle(axis, 2 * np.pi / 37)   # order-37 cyclic subgroup

p = sample_sphere(1, 3, rng)[0]
orbit = [p]
for _ in range(36):
    orbit.append(R_small @ orbit[-1])
orbit = np.array(orbit)

# a cap around p, carried along by the same subgroup
cap_local = p + 0.22 * rng.normal(size=(40, 3))
cap_local /= np.linalg.norm(cap_local, axis=1, keepdims=True)
cap_orbit = np.concatenate([cap_local @ np.linalg.matrix_power(R_small, k).T
                            for k in range(37)])

fig = plt.figure(figsize=(9.4, 4.2))
ax = sphere_ax(fig, 121, r"orbit of a point under $\langle R\rangle \cong \mathbb{Z}/37$", elev=18)
draw_wire_sphere(ax, alpha=0.10)
ax.plot([0, axis[0] * 1.25], [0, axis[1] * 1.25], [0, axis[2] * 1.25],
        color=GEO_DARK, lw=2)
ax.scatter(*orbit.T, s=22, c=GEO_RUST, depthshade=False)
ax.scatter(*p, s=70, c=GEO_DARK, depthshade=False, marker="*")

ax = sphere_ax(fig, 122, "orbit of a cap: the leakage hazard", elev=18)
draw_wire_sphere(ax, alpha=0.10)
ax.scatter(*cap_orbit.T, s=6, c=GEO_TEAL, alpha=0.7, lw=0)
ax.scatter(*cap_local.T, s=10, c=GEO_RUST, depthshade=False)
plt.tight_layout(); plt.show()

> **Exercise 5 — split by orbit, not by sample.** *(the most important exercise here)*
>
> Build a dataset with a deliberate symmetry: take $M = 200$ "base" point clouds of
> $50$ points each, sampled from a spherical cap of random centre and radius, and
> give each a label $y$ = the cap's angular radius. Then *augment*: replace each
> cloud by $8$ randomly rotated copies, all carrying the same label. You now have
> $1600$ samples in $200$ orbits.
>
> (a) Extract a rotation-**invariant** feature vector (e.g. the sorted pairwise
> geodesic-distance quantiles) and fit a linear model with a random $80/20$ split.
> Record the test error.
>
> (b) Repeat with a *non*-invariant feature vector (e.g. the raw coordinates of the
> cloud's centroid). Record the test error.
>
> (c) Now redo (b) splitting by **orbit** — all $8$ copies of a base cloud go to the
> same side of the split. Compare the three test errors.
>
> You should find that (b) looks excellent and is a lie: the model has seen a rotated
> copy of every test cloud during training. This is not a contrived failure mode; it
> is one of the commonest ways published computational results turn out to be wrong.

In [ ]:
# Construção das 200 nuvens-base e das 8 cópias rotacionadas de cada uma.
M_EX5, POINTS_EX5, COPIES_EX5 = 200, 50, 8
rng_ex5 = np.random.default_rng(SEED + 5)


def sample_cap(center, radius, n_points, rng):
    center = center / np.linalg.norm(center)
    pole = np.array([0.0, 0.0, 1.0])
    if np.abs(center @ pole) > 0.95:
        pole = np.array([1.0, 0.0, 0.0])
    tangent = pole - (pole @ center) * center
    tangent /= np.linalg.norm(tangent)
    other = np.cross(center, tangent)
    azimuth = rng.uniform(0, 2 * np.pi, n_points)
    cos_angle = rng.uniform(np.cos(radius), 1.0, n_points)
    sin_angle = np.sqrt(1.0 - cos_angle**2)
    return (cos_angle[:, None] * center
            + sin_angle[:, None] * (np.cos(azimuth)[:, None] * tangent
                                    + np.sin(azimuth)[:, None] * other))


base_clouds, base_labels = [], []
for _ in range(M_EX5):
    center = sample_sphere(1, 3, rng_ex5)[0]
    radius = rng_ex5.uniform(0.18, 0.85)
    base_clouds.append(sample_cap(center, radius, POINTS_EX5, rng_ex5))
    base_labels.append(radius)

clouds, labels, orbit_ids = [], [], []
for orbit_id, (cloud, label) in enumerate(zip(base_clouds, base_labels)):
    for _ in range(COPIES_EX5):
        clouds.append(cloud @ random_rotation(rng_ex5).T)
        labels.append(label)
        orbit_ids.append(orbit_id)
clouds = np.array(clouds)
labels = np.array(labels)
orbit_ids = np.array(orbit_ids)

rng_split = np.random.default_rng(SEED + 6)
random_order = rng_split.permutation(len(labels))
random_train = np.zeros(len(labels), dtype=bool)
random_train[random_order[:int(0.8 * len(labels))]] = True
random_test = ~random_train

orbit_order = rng_split.permutation(M_EX5)
train_orbits = set(orbit_order[:int(0.8 * M_EX5)])
orbit_train = np.isin(orbit_ids, list(train_orbits))
orbit_test = ~orbit_train

print("dataset:", clouds.shape, "samples in", len(np.unique(orbit_ids)), "orbits")

In [ ]:
# Calota gerada com centro fixado na direção frontal da visualização.
elev = np.deg2rad(20)
azim = np.deg2rad(35)

center_front = np.array([
    np.cos(elev) * np.cos(azim),
    np.cos(elev) * np.sin(azim),
    np.sin(elev),
])

cap_radius_to_plot = 0.45
cap_to_plot = sample_cap(
    center_front,
    cap_radius_to_plot,
    POINTS_EX5,
    rng_ex5,
)

fig = plt.figure(figsize=(4.8, 4.5))
ax = sphere_ax(
    fig,
    111,
    f"calota uniforme frontal, raio angular = {cap_radius_to_plot:.3f} rad",
    elev=20,
    azim=35,
    lim=1.08,
)
draw_wire_sphere(ax, alpha=0.10)
ax.scatter(*cap_to_plot.T, s=28, c=GEO_TEAL, alpha=0.85, lw=0,
           label="pontos da calota")
ax.scatter(*center_front, s=75, c=GEO_RUST, marker="*", depthshade=False,
           label="centro da calota")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**(a)** Para uma nuvem $C=(x_1,\ldots,x_{50})$, definimos $q(C)$ como o vetor formado pelos quantis $0{,}10$, $0{,}25$, $0{,}50$, $0{,}75$ e $0{,}90$ do conjunto de distâncias geodésicas

$$\{d_{S^2}(x_i,x_j):1\leq i<j\leq 50\}.$$

Para todo $R\in SO(3)$, vale $d_{S^2}(Rx_i,Rx_j)=d_{S^2}(x_i,x_j)$; logo, $q(RC)=q(C)$. Ajustaremos uma regressão linear de $q(C)$ para o raio angular. O split aleatório, entretanto, não produz amostras independentes no nível das órbitas: cópias rotacionadas de uma mesma nuvem-base podem aparecer simultaneamente no treino e no teste. Como uma regressão linear de baixa dimensão pode não explorar essa repetição de maneira visível, isolaremos o efeito do split na parte (c) com um estimador capaz de reconhecer cópias equivalentes.

In [ ]:
def invariant_features(clouds):
    features = []
    for cloud in clouds:
        gram = np.clip(cloud @ cloud.T, -1.0, 1.0)
        pairwise_distances = np.arccos(gram[np.triu_indices(len(cloud), k=1)])
        features.append(np.quantile(pairwise_distances, [0.1, 0.25, 0.5, 0.75, 0.9]))
    return np.array(features)


def fit_and_predict(features, targets, train_mask, test_mask):
    design_train = np.column_stack([np.ones(train_mask.sum()), features[train_mask]])
    coefficients, *_ = np.linalg.lstsq(design_train, targets[train_mask], rcond=None)
    design_test = np.column_stack([np.ones(test_mask.sum()), features[test_mask]])
    return design_test @ coefficients


features_inv = invariant_features(clouds)
pred_inv_random = fit_and_predict(features_inv, labels, random_train, random_test)
error_inv_random = np.sqrt(np.mean((pred_inv_random - labels[random_test])**2))

fig, ax = plt.subplots(figsize=(5.0, 3.4))
ax.scatter(labels[random_test], pred_inv_random, s=10, alpha=0.45, color=GEO_TEAL)
ax.plot([labels.min(), labels.max()], [labels.min(), labels.max()], "--", color=GEO_RUST)
ax.set_xlabel("raio angular verdadeiro")
ax.set_ylabel("previsão")
ax.set_title(f"(a) quantis invariantes, split aleatório: RMSE = {error_inv_random:.4f}")
plt.tight_layout(); plt.show()

**(b)** Usaremos agora o centroide empírico

$$\bar x(C)=\frac1{50}\sum_{i=1}^{50}x_i\in\mathbb R^3.$$

Essa representação é equivariante, e não invariante: $\bar x(RC)=R\bar x(C)$. Para uma calota uniformemente distribuída, de centro $c\in S^2$ e raio angular $\alpha$, o centroide populacional satisfaz

$$\mathbb E[X]=\frac{1+\cos\alpha}{2}\,c.$$

Assim, a norma do centroide contém a informação sobre $\alpha$, mas essa dependência é não linear e invariante por rotação. Uma função linear das três coordenadas do centroide não pode ser invariante sob todo $SO(3)$, salvo se for constante. Portanto, não se espera que a regressão linear abaixo reconheça bem o raio; esta parte mede também a inadequação da representação linear, e não apenas o efeito do split.

In [ ]:
def centroid_features(clouds):
    return clouds.mean(axis=1)


features_centroid = centroid_features(clouds)
pred_centroid_random = fit_and_predict(
    features_centroid, labels, random_train, random_test
)
error_centroid_random = np.sqrt(
    np.mean((pred_centroid_random - labels[random_test])**2)
)

fig, ax = plt.subplots(figsize=(5.0, 3.4))
ax.scatter(labels[random_test], pred_centroid_random, s=10, alpha=0.45, color=GEO_TEAL)
ax.plot([labels.min(), labels.max()], [labels.min(), labels.max()], "--", color=GEO_RUST)
ax.set_xlabel("raio angular verdadeiro")
ax.set_ylabel("previsão")
ax.set_title(
    f"(b) centroide não invariante, split aleatório: RMSE = {error_centroid_random:.4f}"
)
plt.tight_layout(); plt.show()

**(c)** Repetimos agora exatamente o procedimento da parte (b): usamos as mesmas três coordenadas do centroide como feature não invariante e ajustamos o mesmo modelo linear. A única alteração é a partição dos dados.

Uma órbita é o conjunto das oito cópias rotacionadas de uma mesma nuvem-base. No split por órbita, escolhemos $160$ das $200$ órbitas para treino e reservamos as outras $40$ para teste; portanto, todas as oito cópias de uma nuvem-base pertencem ao mesmo lado da partição. Assim, a comparação entre (b) e (c) mantém fixos a representação e o estimador e altera somente o procedimento de split.

In [ ]:
pred_centroid_orbit = fit_and_predict(
    features_centroid, labels, orbit_train, orbit_test
)
error_centroid_orbit = np.sqrt(
    np.mean((pred_centroid_orbit - labels[orbit_test])**2)
)

fig, axes = plt.subplots(1, 2, figsize=(8.0, 3.2), sharex=True, sharey=True)
panels = [
    (labels[random_test], pred_centroid_random,
     f"(b) centroide, split aleatório\nRMSE = {error_centroid_random:.4f}"),
    (labels[orbit_test], pred_centroid_orbit,
     f"(c) centroide, split por órbita\nRMSE = {error_centroid_orbit:.4f}"),
]
for ax, (truth, prediction, title) in zip(axes, panels):
    ax.scatter(truth, prediction, s=9, alpha=0.4, color=GEO_TEAL)
    ax.plot([labels.min(), labels.max()], [labels.min(), labels.max()],
            "--", color=GEO_RUST)
    ax.set_xlabel("raio angular verdadeiro")
    ax.set_title(title)
axes[0].set_ylabel("previsão")
fig.suptitle("Mesma feature e mesmo modelo; apenas o split muda", y=1.03)
plt.tight_layout(); plt.show()

print("órbitas no treino:", len(np.unique(orbit_ids[orbit_train])))
print("órbitas no teste:", len(np.unique(orbit_ids[orbit_test])))
print("RMSE centroide linear, split aleatório:", f"{error_centroid_random:.4f}")
print("RMSE centroide linear, split por órbita:", f"{error_centroid_orbit:.4f}")

**Diagnóstico adicional — tornando o vazamento diretamente observável.** Este experimento não substitui a parte (c). Agora mantemos fixos os quantis invariantes $q(C)$ da parte (a) e o estimador $1$-NN. Dado um vetor de features $z$, definimos

$$\widehat y(z)=y_{i^*(z)},\qquad i^*(z)\in\operatorname*{argmin}_{i\in\mathcal I_{\mathrm{train}}}\|z-z_i\|_2.$$

Se uma cópia da mesma órbita estiver no treino, então, a menos de erro de arredondamento, sua distância ao vetor de teste é zero e seu rótulo é o mesmo. No split por órbita, nenhuma coincidência dessa natureza está disponível.

In [ ]:
def nearest_neighbour_predict(features, targets, train_mask, test_mask):
    tree = cKDTree(features[train_mask])
    _, nearest = tree.query(features[test_mask], k=1)
    return targets[train_mask][nearest]


pred_nn_random = nearest_neighbour_predict(
    features_inv, labels, random_train, random_test
)
pred_nn_orbit = nearest_neighbour_predict(
    features_inv, labels, orbit_train, orbit_test
)
error_nn_random = np.sqrt(np.mean((pred_nn_random - labels[random_test])**2))
error_nn_orbit = np.sqrt(np.mean((pred_nn_orbit - labels[orbit_test])**2))
test_orbit_seen = np.isin(orbit_ids[random_test], orbit_ids[random_train])

fig, axes = plt.subplots(1, 2, figsize=(8.0, 3.2), sharex=True, sharey=True)
panels = [
    (labels[random_test], pred_nn_random,
     f"1-NN invariante, split aleatório\nRMSE = {error_nn_random:.4e}"),
    (labels[orbit_test], pred_nn_orbit,
     f"1-NN invariante, split por órbita\nRMSE = {error_nn_orbit:.4f}"),
]
for ax, (truth, prediction, title) in zip(axes, panels):
    ax.scatter(truth, prediction, s=9, alpha=0.4, color=GEO_TEAL)
    ax.plot([labels.min(), labels.max()], [labels.min(), labels.max()],
            "--", color=GEO_RUST)
    ax.set_xlabel("raio angular verdadeiro")
    ax.set_title(title)
axes[0].set_ylabel("previsão")
fig.suptitle("Diagnóstico adicional de vazamento entre órbitas", y=1.03)
plt.tight_layout(); plt.show()

print("fração dos testes aleatórios cuja órbita aparece no treino:",
      f"{test_orbit_seen.mean():.4f}")
print("RMSE 1-NN invariante, split aleatório:", f"{error_nn_random:.6e}")
print("RMSE 1-NN invariante, split por órbita:", f"{error_nn_orbit:.4f}")

---
## 5. $S^3$: quaternions and the Hopf fibration

$S^3 \subset \mathbb{R}^4$ cannot be drawn. This is the generic situation in machine
learning — your data lives in $\mathbb{R}^{784}$ or $\mathbb{R}^{10^6}$ — so it is
worth practising the three standard responses on a case where we know the truth.

1. **Project to coordinate subspaces.** Cheap, and almost always misleading.
2. **Use a global chart.** Stereographic projection $S^3 \setminus \{N\} \to \mathbb{R}^3$
   is a diffeomorphism, conformal, and distorts only scale.
3. **Use the extra structure.** $S^3$ is a Lie group (unit quaternions) and fibres
   over $S^2$ with circle fibres — the Hopf fibration. Colouring by the fibre gives
   a genuinely faithful picture.

In [ ]:
Q = sample_sphere(4000, 4, rng)
print("samples on S^3 :", Q.shape, " norms all 1 :",
      np.allclose(np.linalg.norm(Q, axis=1), 1.0))

fig, axes = plt.subplots(1, 3, figsize=(10.0, 3.2))
for ax, (i, j) in zip(axes, [(0, 1), (0, 2), (2, 3)]):
    ax.scatter(Q[:, i], Q[:, j], s=2, c=GEO_TEAL, alpha=0.35, lw=0)
    ax.set_aspect("equal"); ax.set_xlabel(f"$q_{i+1}$"); ax.set_ylabel(f"$q_{j+1}$")
fig.suptitle("Coordinate projections of $S^3$: filled discs, no structure visible", y=1.03)
plt.tight_layout(); plt.show()

The projections are uninformative — each is a filled disc, because the projection of
the uniform measure on $S^3$ to two coordinates has density
$\propto (1 - x^2 - y^2)^{0}$, i.e. it is *exactly uniform on the unit disc*.
(Check this: it is the $n=4$ case of the marginal formula in §6.)

### The Hopf map

Identify $\mathbb{R}^4 \cong \mathbb{C}^2$ by $q = (a,b,c,d) \mapsto (z_1, z_2) = (a+ib,\, c+id)$.
The Hopf map is

$$h(z_1,z_2) = \big(2\,\mathrm{Re}(z_1\bar z_2),\; 2\,\mathrm{Im}(z_1 \bar z_2),\; |z_1|^2 - |z_2|^2\big) \in S^2 .$$

Its fibres are the orbits of the $U(1)$ action $(z_1,z_2) \mapsto (e^{i\tau}z_1, e^{i\tau}z_2)$:
great circles in $S^3$, pairwise linked, and $S^3$ is their disjoint union. This is
the reason $\pi_3(S^2) = \mathbb{Z}$.

In [ ]:
def hopf_map(Q):
    """S^3 -> S^2, the Hopf fibration."""
    a, b, c, d = Q[:, 0], Q[:, 1], Q[:, 2], Q[:, 3]
    return np.stack([2 * (a * c + b * d),
                     2 * (b * c - a * d),
                     a * a + b * b - c * c - d * d], axis=1)


def hopf_section(p):
    """One preimage in S^3 of a point p in S^2 (undefined at the south pole)."""
    x, y, z = p
    q0 = np.array([1 + z, 0.0, x, -y])
    return q0 / np.linalg.norm(q0)


def hopf_fibre(p, n=400):
    """The full circle fibre h^{-1}(p) subset S^3."""
    a, b, c, d = hopf_section(p)
    tau = np.linspace(0, 2 * np.pi, n)
    return np.stack([a * np.cos(tau) - b * np.sin(tau),
                     a * np.sin(tau) + b * np.cos(tau),
                     c * np.cos(tau) - d * np.sin(tau),
                     c * np.sin(tau) + d * np.cos(tau)], axis=1)


# verification: the image is on S^2, and the fibre really is a fibre
B = hopf_map(Q)
assert np.allclose(np.linalg.norm(B, axis=1), 1.0)
p_test = sample_sphere(1, 3, rng)[0]
F = hopf_fibre(p_test)
assert np.allclose(np.linalg.norm(F, axis=1), 1.0)
assert np.allclose(hopf_map(F), p_test, atol=1e-10)
print("Hopf map lands on S^2 and its fibres are constant — verified")

# the Hopf map pushes uniform S^3 forward to uniform S^2 (Archimedes test again)
print("mean of z-coordinate of h(Q):", B[:, 2].mean().round(4), " (should be ~0)")

### Stereographic projection, and the fibres as linked circles

Project $S^3 \setminus \{(0,0,0,1)\} \to \mathbb{R}^3$ by
$q \mapsto (q_1,q_2,q_3)/(1-q_4)$. Circles not through the projection pole map to
circles in $\mathbb{R}^3$. Fibres over a circle of latitude in $S^2$ therefore sweep
out a torus, and fibres over different latitudes give **nested tori** filling
$\mathbb{R}^3$ — the classical picture of $S^3$ as two solid tori glued along their
boundary.

In [ ]:
def stereographic(Q):
    """S^3 \\ {e_4} -> R^3."""
    return Q[:, :3] / (1 - Q[:, 3:4])


def latitude_circle(z0, n):
    """n points on the circle {height = z0} of S^2."""
    t = np.linspace(0, 2 * np.pi, n, endpoint=False)
    r = np.sqrt(1 - z0**2)
    return np.stack([r * np.cos(t), r * np.sin(t), np.full_like(t, z0)], axis=1)


def hopf_ax(fig, pos, title, lim, elev=58):
    ax = fig.add_subplot(pos, projection="3d")
    ax.set_title(title, fontsize=9)
    ax.set_box_aspect((1, 1, 1))
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_zlim(-lim, lim)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
    ax.grid(False); ax.view_init(elev=elev, azim=30)
    return ax


fig = plt.figure(figsize=(11.6, 4.3))
cmap = plt.get_cmap("viridis")

# (a) all fibres over ONE latitude sweep out a single torus
ax = hopf_ax(fig, 131, "fibres over one latitude ($z=0.5$)\nsweep out a torus", 1.85)
for k, p_base in enumerate(latitude_circle(0.5, 32)):
    P = stereographic(hopf_fibre(p_base, 400))
    ax.plot(*P.T, lw=0.7, color=cmap(k / 32), alpha=0.85)

# (b) different latitudes give nested tori filling R^3
ax = hopf_ax(fig, 132, "three latitudes\n$\\Rightarrow$ nested tori", 2.5)
for z0, col, a, m in [(0.9, GEO_DARK, 0.95, 9), (0.5, GEO_TEAL, 0.7, 13),
                      (0.0, GEO_RUST, 0.45, 17)]:
    for p_base in latitude_circle(z0, m):
        P = stereographic(hopf_fibre(p_base, 400))
        ax.plot(*P.T, lw=0.8, color=col, alpha=a)

# (c) any two fibres are linked exactly once
ax = hopf_ax(fig, 133, "any two distinct fibres\nare linked (Hopf link)", 1.85, elev=42)
for p_base, col in zip(latitude_circle(0.5, 3), [GEO_DARK, GEO_TEAL, GEO_RUST]):
    P = stereographic(hopf_fibre(p_base, 600))
    ax.plot(*P.T, lw=2.4, color=col)

plt.tight_layout(); plt.show()

### A faithful 2-D picture of $S^3$

Finally, a picture that uses the fibration to encode all four dimensions honestly:
plot the base point $h(q) \in S^2$ in a Mollweide (equal-area) projection, and
colour by the fibre coordinate $\tau$. Every point of $S^3$ has a unique position in
this diagram, and equal areas of the diagram carry equal measure.

In [ ]:
def fibre_phase(Q):
    """The U(1) coordinate along the fibre, as an angle in (-pi, pi]."""
    return np.arctan2(Q[:, 1], Q[:, 0])


B = hopf_map(Q)
lon = np.arctan2(B[:, 1], B[:, 0])
lat = np.arcsin(np.clip(B[:, 2], -1, 1))

fig = plt.figure(figsize=(7.6, 3.9))
ax = fig.add_subplot(111, projection="mollweide")
sc = ax.scatter(lon, lat, c=fibre_phase(Q), cmap="twilight", s=5, alpha=0.85, lw=0)
ax.grid(alpha=0.3); ax.set_xticklabels([]); ax.tick_params(labelsize=6)
ax.set_title("$S^3$ drawn as (base point in $S^2$) $\\times$ (fibre phase, colour)", pad=14)
plt.colorbar(sc, ax=ax, shrink=0.72, label=r"$\tau$")
plt.tight_layout(); plt.show()

> **Exercise 6 — $S^3$ hands-on.**
> (a) The unit quaternions form a group. Implement quaternion multiplication and
> check numerically that `quat_to_rotation` is a homomorphism onto $SO(3)$:
> $R(q_1 q_2) = R(q_1) R(q_2)$. Confirm the kernel is $\{\pm 1\}$.
>
> (b) Verify the Hopf fibre through $q$ is exactly the orbit of $q$ under
> right-multiplication by the unit complex numbers, and that distinct fibres are
> disjoint (sample two nearby base points and compute the minimum distance between
> their fibres).
>
> (c) Two distinct fibres are *linked*. Compute their linking number numerically
> from the Gauss integral, or argue it from the picture. What does this say about
> $\pi_3(S^2)$?
>
> (d) *(challenge)* $\mathbb{RP}^3 \cong SO(3)$ is $S^3$ with antipodes identified.
> Sample $SO(3)$ uniformly, map each rotation to the *closer* of its two
> quaternion lifts, and visualise the result in the Mollweide diagram above. What
> changes?

**Minha Resposta — (a)** Um quaternion unitário $q=(w,x,y,z)$ representa uma rotação de $\mathbb{R}^3$. O produto de Hamilton combina duas rotações. Se $q_1$ e $q_2$ são unitários, esperamos

$$R(q_1q_2)=R(q_1)R(q_2).$$

Além disso, $q$ e $-q$ representam a mesma rotação; portanto, o núcleo da aplicação $S^3\to SO(3)$ é $\{1,-1\}$.

In [ ]:
def quat_multiply(q1, q2):
    """Hamilton product for quaternions written as (w, x, y, z)."""
    w1, x1, y1, z1 = q1
    w2, x2, y2, z2 = q2
    return np.array([
        w1 * w2 - x1 * x2 - y1 * y2 - z1 * z2,
        w1 * x2 + x1 * w2 + y1 * z2 - z1 * y2,
        w1 * y2 - x1 * z2 + y1 * w2 + z1 * x2,
        w1 * z2 + x1 * y2 - y1 * x2 + z1 * w2,
    ])


q1 = sample_sphere(1, 4, rng)[0]
q2 = sample_sphere(1, 4, rng)[0]
q_product = quat_multiply(q1, q2)

print("product remains unit:", np.allclose(np.linalg.norm(q_product), 1.0))
print("homomorphism verified:",
      np.allclose(quat_to_rotation(q_product),
                  quat_to_rotation(q1) @ quat_to_rotation(q2), atol=1e-12))
print("kernel identification verified:",
      np.allclose(quat_to_rotation(q1), quat_to_rotation(-q1)))

**(b)** A fibra sobre $p\in S^2$ é o conjunto de pontos de $S^3$ que o mapa de Hopf envia em $p$. Nesta convenção $(a,b,c,d)$, a fórmula existente para `hopf_fibre` é obtida multiplicando **à esquerda** por um número complexo unitário $(\cos\tau,\sin\tau,0,0)$. Essa ação gira simultaneamente os pares $(a,b)$ e $(c,d)$ e deixa a imagem de Hopf constante. A célula seguinte compara numericamente a órbita com a fibra.

In [ ]:
q = sample_sphere(1, 4, rng)[0]
tau = np.linspace(0, 2 * np.pi, 400)
complex_orbit = np.array([
    quat_multiply(np.array([np.cos(angle), np.sin(angle), 0.0, 0.0]), q)
    for angle in tau
])

p_from_q = hopf_map(q[None, :])[0]
images = hopf_map(complex_orbit)
print("orbit stays on S^3:",
      np.allclose(np.linalg.norm(complex_orbit, axis=1), 1.0))
print("Hopf image stays constant:",
      np.allclose(images, p_from_q, atol=1e-10))
print("maximum variation of the Hopf image:",
      f"{np.max(np.linalg.norm(images - p_from_q, axis=1)):.2e}")

**(c)** Depois da projeção estereográfica, duas fibras distintas tornam-se curvas fechadas em $\mathbb{R}^3$. O número de ligação pode ser estimado pela integral de Gauss

$$\operatorname{Link}(C_1,C_2)=\frac{1}{4\pi}\oint\!\oint\frac{(r_1-r_2)\cdot(dr_1\times dr_2)}{\|r_1-r_2\|^3}.$$

Para fibras de Hopf distintas, o resultado deve ser $\pm1$ (o sinal depende das orientações). Isto significa que quaisquer duas fibras estão ligadas uma vez, que é a característica geométrica central da fibragem de Hopf.

In [ ]:
def gauss_linking_number(curve_a, curve_b):
    segment_a = np.roll(curve_a, -1, axis=0) - curve_a
    segment_b = np.roll(curve_b, -1, axis=0) - curve_b
    midpoint_a = 0.5 * (curve_a + np.roll(curve_a, -1, axis=0))
    midpoint_b = 0.5 * (curve_b + np.roll(curve_b, -1, axis=0))
    separation = midpoint_a[:, None, :] - midpoint_b[None, :, :]
    cross_terms = np.cross(segment_a[:, None, :], segment_b[None, :, :])
    denominator = np.linalg.norm(separation, axis=2) ** 3
    return np.sum(np.einsum("ijk,ijk->ij", separation, cross_terms) / denominator) / (4 * np.pi)


base_points = latitude_circle(0.5, 2)
fibre_a = stereographic(hopf_fibre(base_points[0], n=500)[:-1])
fibre_b = stereographic(hopf_fibre(base_points[1], n=500)[:-1])
link_number = gauss_linking_number(fibre_a, fibre_b)

fig = plt.figure(figsize=(5.0, 4.5))
ax = hopf_ax(fig, 111, "duas fibras de Hopf após projeção estereográfica", 1.85, elev=42)
ax.plot(*fibre_a.T, color=GEO_TEAL, lw=2.0, label="fibra 1")
ax.plot(*fibre_b.T, color=GEO_RUST, lw=2.0, label="fibra 2")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
print(f"número de ligação aproximado: {link_number:.4f}")

**(d)** Em $\mathbb{RP}^3\cong SO(3)$ identificamos $q$ e $-q$. Uma escolha de representante é manter somente o levantamento com componente escalar $w\geq0$, equivalente ao quaternion mais próximo da identidade. A visualização deve mostrar apenas uma metade de $S^3$: pontos antipodais não aparecem mais como elementos distintos.

In [ ]:
dims_sweep = [2, 3, 5, 10, 20, 50, 100, 300, 1000]
M = 500
mean_d, spread, mean_absdot = [], [], []
for n in dims_sweep:
    Y = sample_sphere(M, n, rng)
    G = np.clip(Y @ Y.T, -1, 1)
    iu_ = np.triu_indices(M, k=1)
    d = np.sqrt(np.maximum(0, 2 - 2 * G[iu_]))
    mean_d.append(d.mean())
    spread.append((d.max() - d.min()) / d.mean())
    mean_absdot.append(np.abs(G[iu_]).mean())

fig, axes = plt.subplots(1, 3, figsize=(11.0, 3.0))

axes[0].semilogx(dims_sweep, mean_d, "o-", color=GEO_DARK)
axes[0].axhline(np.sqrt(2), color=GEO_RUST, ls="--", lw=1.2, label=r"$\sqrt{2}$")
axes[0].set_xlabel("$n$"); axes[0].set_ylabel("mean pairwise distance")
axes[0].set_title("distances converge"); axes[0].legend(fontsize=8)

axes[1].loglog(dims_sweep, spread, "o-", color=GEO_DARK)
axes[1].set_xlabel("$n$"); axes[1].set_ylabel(r"$(d_{\max}-d_{\min})/\bar d$")
axes[1].set_title("relative spread collapses")

axes[2].loglog(dims_sweep, mean_absdot, "o-", color=GEO_DARK, label="measured")
axes[2].loglog(dims_sweep, np.sqrt(2 / (np.pi * np.array(dims_sweep))),
               "--", color=GEO_RUST, lw=1.2, label=r"$\sqrt{2/\pi n}$")
axes[2].set_xlabel("$n$"); axes[2].set_ylabel(r"$\mathbb{E}\,|\langle x,y\rangle|$")
axes[2].set_title("random vectors are nearly orthogonal"); axes[2].legend(fontsize=8)

plt.tight_layout(); plt.show()

### The blessing

The same phenomenon that destroys nearest-neighbour intuition is what makes
learning possible at all: a Lipschitz function on $S^{n-1}$ satisfies

$$\mathbb{P}\big(|f - m_f| > t\big) \;\leq\; 2 e^{-c\, n t^2},$$

so a Monte-Carlo average over a modest sample estimates its mean to accuracy
$O(1/\sqrt n)$, *independently of the ambient complexity*. Empirical risk is a
Lipschitz function of the sample; concentration is the reason
$\widehat{\mathcal{R}}_N \approx \mathcal{R}$ at all. Let us watch it happen.

In [ ]:
f_lip = lambda X: X[:, 0]     # 1-Lipschitz on the sphere

fig, ax = plt.subplots(figsize=(5.6, 3.0))
for n, col in zip([3, 30, 300, 3000], [GEO_RUST, GEO_TEAL, GEO_DARK, "#4c9a2a"]):
    vals = f_lip(sample_sphere(20000, n, rng))
    ax.hist(vals, bins=80, range=(-1, 1), density=True, histtype="step",
            lw=1.6, color=col, label=f"$n={n}$")
ax.set_xlim(-1, 1); ax.set_xlabel("$f(x) = x_1$"); ax.set_ylabel("density")
ax.set_title("concentration of a 1-Lipschitz function about its median")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

> **Exercise 7 — quantifying the curse.**
> (a) Verify the sub-Gaussian bound directly: for each $n$, estimate
> $\mathbb{P}(|x_1| > t)$ by simulation and plot $-\log \mathbb{P}$ against $n t^2$.
> Do the curves collapse onto a line? What constant $c$ do you measure?
>
> (b) *Sample complexity.* To cover $S^{n-1}$ so that every point is within
> geodesic distance $\varepsilon$ of a sample, roughly how many i.i.d. points are
> needed? Measure the covering radius of an $N$-point sample as a function of $N$
> for $n = 3, 5, 10$ and compare with the theoretical $N \sim \varepsilon^{-(n-1)}$.
>
> (c) *The escape.* Repeat (b) for points sampled from a fixed $2$-dimensional
> great subsphere $S^2 \subset S^{n-1}$. The covering radius should now be governed
> by the *intrinsic* dimension $2$, not by $n$. This single experiment is the
> content of the manifold hypothesis — and the subject of the next section.

**Minha Resposta:**

**(a)** Se $X$ é uniforme em $S^{n-1}$, então $\sqrt{n}\,x_1$ converge em distribuição para $\mathcal{N}(0,1)$. Logo, no regime de cauda,

$$\mathbb{P}(|x_1|>t) \approx 2\Phi(-\sqrt{n}t)
\quad\Longrightarrow\quad
-\log \mathbb{P}(|x_1|>t)=\frac{nt^2}{2}+O(\log(\sqrt{n}t)).$$

Portanto, as curvas devem quase colapsar quando escritas em função de $nt^2$. A inclinação assintótica, isto é, a constante sub-Gaussiana medida, deve ser $c\approx 1/2$. Em uma faixa finita da cauda, o termo logarítmico faz o ajuste produzir uma inclinação um pouco maior que $1/2$.

In [ ]:
rng_ex7_a = np.random.default_rng(SEED + 7)
dims_tail = [30, 100, 300, 1000]
scaled_thresholds = np.linspace(1.0, 12.0, 18)  # valores de n t^2
n_tail_samples = 500_000

fig, ax = plt.subplots(figsize=(6.0, 3.5))
fit_x, fit_y = [], []

for n in dims_tail:
    # Simulação de Muller sem armazenar as n coordenadas que não serão usadas.
    g1 = rng_ex7_a.normal(size=n_tail_samples)
    remaining_norm_squared = rng_ex7_a.chisquare(n - 1, size=n_tail_samples)
    abs_x1 = np.abs(g1) / np.sqrt(g1**2 + remaining_norm_squared)
    thresholds = np.sqrt(scaled_thresholds / n)
    counts = np.array([(abs_x1 > t).sum() for t in thresholds])
    # A correção 1/2 evita log(0) caso nenhuma observação caia na cauda.
    probabilities = (counts + 0.5) / (n_tail_samples + 1.0)
    negative_log_probability = -np.log(probabilities)
    ax.plot(scaled_thresholds, negative_log_probability, "o-", ms=3, label=rf"$n={n}$")

    tail = scaled_thresholds >= 6.0
    if n >= 100:
        fit_x.extend(scaled_thresholds[tail])
        fit_y.extend(negative_log_probability[tail])

c_measured, intercept = np.polyfit(fit_x, fit_y, 1)
x_line = np.array([scaled_thresholds.min(), scaled_thresholds.max()])
ax.plot(x_line, c_measured * x_line + intercept, "--", color=GEO_RUST, lw=1.8,
        label=rf"ajuste na cauda: $c={c_measured:.3f}$")
ax.set_xlabel(r"$nt^2$"); ax.set_ylabel(r"$-\log \widehat{\mathbb{P}}(|x_1|>t)$")
ax.set_title("Colapso das caudas em escala sub-Gaussiana")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
print(f"constante medida na cauda: c = {c_measured:.3f} (limite assintótico: 0.5)")

**(b)** Escreva $d=n-1=\dim S^{n-1}$. Para $\varepsilon$ pequeno, uma bola geodésica de raio $\varepsilon$ ocupa uma fração $\asymp \varepsilon^d$ da esfera. Assim, são necessários aproximadamente

$$N\asymp \varepsilon^{-d}=\varepsilon^{-(n-1)},
\qquad\text{ou, equivalentemente,}\qquad
\varepsilon_N\asymp N^{-1/(n-1)}.$$

Como a amostra é i.i.d., aparecem buracos aleatórios e o resultado mais preciso contém a correção $\varepsilon_N\asymp(\log N/N)^{1/d}$. No experimento, aproximamos o máximo sobre toda a esfera pelo máximo sobre um conjunto grande e independente de pontos-testes; portanto, os valores obtidos são estimativas inferiores do verdadeiro raio de cobertura.

In [ ]:
def empirical_covering_radius(sample, probes):
    """Maior distância geodésica de um ponto-teste à amostra."""
    nearest_chordal, _ = cKDTree(sample).query(probes, k=1)
    return 2 * np.arcsin(np.clip(nearest_chordal.max() / 2, 0.0, 1.0))


def estimate_covering_curve(sampler, sizes, rng, n_probes=15_000, n_repeats=3):
    radii = np.empty((n_repeats, len(sizes)))
    for repeat in range(n_repeats):
        # Usar amostras aninhadas reduz a oscilação da curva quando N cresce.
        cloud = sampler(int(sizes[-1]), rng)
        probes = sampler(n_probes, rng)
        for j, n_points in enumerate(sizes):
            radii[repeat, j] = empirical_covering_radius(cloud[:n_points], probes)
    return radii.mean(axis=0), radii.std(axis=0)


rng_ex7_b = np.random.default_rng(SEED + 8)
cover_sizes = np.array([50, 100, 200, 500, 1000, 2000, 5000])
dims_cover = [3, 5, 10]
covering_full = {}
fig, ax = plt.subplots(figsize=(6.2, 3.8))

for n, marker in zip(dims_cover, ["o", "s", "^"]):
    sampler = lambda m, generator, dim=n: sample_sphere(m, dim, generator)
    mean_radius, std_radius = estimate_covering_curve(sampler, cover_sizes, rng_ex7_b)
    covering_full[n] = (mean_radius, std_radius)
    line, = ax.loglog(cover_sizes, mean_radius, marker + "-", ms=4,
                      label=rf"$S^{{{n-1}}}$: medido")
    ax.fill_between(cover_sizes, mean_radius - std_radius, mean_radius + std_radius,
                    color=line.get_color(), alpha=0.12)
    reference = mean_radius[0] * (cover_sizes / cover_sizes[0])**(-1 / (n - 1))
    ax.loglog(cover_sizes, reference, "--", color=line.get_color(), alpha=0.75,
              label=rf"referência $N^{{-1/{n-1}}}$")
    measured_power = np.polyfit(np.log(cover_sizes), np.log(mean_radius), 1)[0]
    print(f"n={n:2d}: expoente medido {measured_power:.3f}; "
          f"expoente teórico {-1/(n-1):.3f}")

ax.set_xlabel(r"número de amostras $N$")
ax.set_ylabel(r"raio de cobertura empírico $\varepsilon_N$")
ax.set_title("A cobertura sofre com a dimensão")
ax.legend(fontsize=7, ncol=2)
plt.tight_layout(); plt.show()

**(c)** Considere a grande subesfera

$$S^2=\{(x_1,x_2,x_3,0,\ldots,0)\in\mathbb{R}^n:\ x_1^2+x_2^2+x_3^2=1\}.$$

Sua métrica geodésica e o volume de suas bolas não dependem da quantidade de coordenadas nulas. Agora $d=2$, qualquer que seja o $n$ ambiente; logo, $N\asymp\varepsilon^{-2}$ e $\varepsilon_N\asymp N^{-1/2}$ (novamente, a menos da correção logarítmica para amostras i.i.d.). A dimensão que controla a complexidade amostral é, portanto, a dimensão intrínseca dos dados, não a do espaço onde eles foram representados.

In [ ]:
def sample_great_s2(n_points, ambient_dim, rng):
    points = np.zeros((n_points, ambient_dim))
    points[:, :3] = sample_sphere(n_points, 3, rng)
    return points


rng_ex7_c = np.random.default_rng(SEED + 9)
ambient_dims = [3, 10, 50]
fig, ax = plt.subplots(figsize=(6.2, 3.8))
great_s2_results = {}

for ambient_dim, marker in zip(ambient_dims, ["o", "s", "^"]):
    sampler = lambda m, generator, dim=ambient_dim: sample_great_s2(m, dim, generator)
    mean_radius, std_radius = estimate_covering_curve(sampler, cover_sizes, rng_ex7_c)
    great_s2_results[ambient_dim] = (mean_radius, std_radius)
    line, = ax.loglog(cover_sizes, mean_radius, marker + "-", ms=4,
                      label=rf"$S^2\subset\mathbb{{R}}^{{{ambient_dim}}}$")
    ax.fill_between(cover_sizes, mean_radius - std_radius, mean_radius + std_radius,
                    color=line.get_color(), alpha=0.12)
    measured_power = np.polyfit(np.log(cover_sizes), np.log(mean_radius), 1)[0]
    print(f"dimensão ambiente {ambient_dim:2d}: expoente medido {measured_power:.3f}")

reference_s2 = great_s2_results[3][0][0] * (cover_sizes / cover_sizes[0])**(-0.5)
ax.loglog(cover_sizes, reference_s2, "k--", lw=1.4, label=r"referência $N^{-1/2}$")
ax.loglog(cover_sizes, covering_full[10][0], ":", color=GEO_RUST, lw=2.0,
          label=r"comparação: esfera ambiente $S^9$")
ax.set_xlabel(r"número de amostras $N$")
ax.set_ylabel(r"raio de cobertura empírico $\varepsilon_N$")
ax.set_title("O escape: a dimensão intrínseca controla a cobertura")
ax.legend(fontsize=7, ncol=2)
plt.tight_layout(); plt.show()

---
## 7. The manifold hypothesis: recovering $S^2$ from $\mathbb{R}^{100}$

Lecture 2, slide 8: real data in $\mathbb{R}^n$ concentrates near a submanifold of
dimension $d \ll n$, and methods succeed when they exploit that. We can now stage
the situation exactly, because we control the ground truth.

Take our cloud on $S^2 \subset \mathbb{R}^3$, map it isometrically into
$\mathbb{R}^{100}$ by a random linear isometry $A: \mathbb{R}^3 \to \mathbb{R}^{100}$
(orthonormal columns), and add isotropic noise of size $\sigma$. The result *looks*
100-dimensional. Two standard tools recover the truth.

In [ ]:
def random_isometry(d, n, rng):
    """A linear map R^d -> R^n with orthonormal columns (so it preserves lengths)."""
    Aq, _ = np.linalg.qr(rng.normal(size=(n, d)))
    return Aq                       # shape (n, d)


d_true, n_ambient, sigma = 3, 100, 0.002
N_MAN = 4000

X_man = sample_sphere(N_MAN, 3, rng)          # the true manifold sample
A = random_isometry(d_true, n_ambient, rng)
X_amb = X_man @ A.T + sigma * rng.normal(size=(N_MAN, n_ambient))

print("ambient data shape:", X_amb.shape)
print("length preserved by A :",
      np.allclose(np.linalg.norm(X_man @ A.T, axis=1), 1.0))

### Global structure: PCA

Principal component analysis is nothing but the eigendecomposition of the
covariance — Lecture 8 will treat it properly. Here it should see through the
embedding immediately: the *linear span* of the data is $3$-dimensional, so exactly
three singular values are $O(1)$ and the remaining $97$ are $O(\sigma)$.

In [ ]:
Xc = X_amb - X_amb.mean(axis=0)
sv = np.linalg.svd(Xc, compute_uv=False)
explained = sv**2 / np.sum(sv**2)

fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.0))
axes[0].semilogy(np.arange(1, 21), sv[:20], "o-", color=GEO_DARK)
axes[0].set_xlabel("index"); axes[0].set_ylabel("singular value")
axes[0].set_title("scree plot: a cliff after 3")
axes[0].axvline(3.5, color=GEO_RUST, ls="--", lw=1.2)

axes[1].bar(np.arange(1, 11), explained[:10], color=GEO_TEAL)
axes[1].set_xlabel("component"); axes[1].set_ylabel("fraction of variance")
axes[1].set_title(f"top 3 explain {explained[:3].sum():.4f} of the variance")
plt.tight_layout(); plt.show()

The first three principal directions span (approximately) the image of $A$, so
projecting onto them should return the sphere — up to a rotation of $\mathbb{R}^3$,
which is precisely the ambiguity PCA cannot resolve, and precisely the ambiguity we
do not care about.

In [ ]:
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
X_rec = Xc @ Vt[:3].T                     # coordinates in the top-3 PCA basis

fig = plt.figure(figsize=(9.2, 4.0))
ax = sphere_ax(fig, 121, "original $S^2$", elev=18)
ax.scatter(*X_man[:800].T, s=3, c=GEO_TEAL, alpha=0.6, lw=0)
ax = sphere_ax(fig, 122, "recovered from $\\mathbb{R}^{100}$ by PCA", elev=18, lim=1.15)
ax.scatter(*X_rec[:800].T, s=3, c=GEO_RUST, alpha=0.6, lw=0)
plt.tight_layout(); plt.show()

print("recovered radii: mean %.4f, std %.4f" %
      (np.linalg.norm(X_rec, axis=1).mean(), np.linalg.norm(X_rec, axis=1).std()))

### Local structure: intrinsic dimension by local PCA

PCA found $3$ — the dimension of the ambient *span*, not of the manifold. The
manifold is $2$-dimensional, and to see that we must look locally: near each point,
the data fills a neighbourhood of the tangent plane $T_p S^2$, which is
$2$-dimensional. So run PCA on each point's $k$ nearest neighbours and look at the
local spectrum.

Read the resulting spectrum carefully — it has **three** regimes, and a geometer
should recognise all of them. For a patch of radius $r$:

$$\underbrace{\sigma_1, \sigma_2 \sim r}_{\text{tangent plane } T_pS^2}
\qquad
\underbrace{\sigma_3 \sim \tfrac{1}{2}\kappa r^2}_{\text{second fundamental form}}
\qquad
\underbrace{\sigma_4, \ldots \sim \sigma_{\text{noise}}}_{\text{ambient noise floor}}$$

The third singular value is not an error: it is the *curvature*, measured. It
shrinks quadratically as the neighbourhood shrinks, which is exactly the statement
that $S^2$ is $C^2$-close to its tangent plane. Estimating intrinsic dimension is
therefore a matter of choosing $r$ small enough that $\kappa r^2 \ll r$ and large
enough that $r \gg \sigma_{\text{noise}}$ — and Exercise 8 asks you to find the
window.

In [ ]:
def local_spectra(X, k=40, n_query=600, n_keep=6):
    """Top singular values of each point's k-neighbourhood, centred at the point."""
    tree = cKDTree(X)                       # neighbours drawn from the whole cloud ...
    _, idx = tree.query(X[:n_query], k=k + 1)   # ... but we only need a sample of patches
    out = np.empty((n_query, n_keep))
    for i in range(n_query):
        nb = X[idx[i, 1:]] - X[i]
        out[i] = np.linalg.svd(nb, compute_uv=False)[:n_keep]
    return out


spec = local_spectra(X_amb, k=40)
spec_mean = spec.mean(axis=0)
spec_mean /= spec_mean[0]
print("mean local spectrum:", spec_mean.round(3))

fig = plt.figure(figsize=(9.6, 3.4))
ax0 = fig.add_subplot(121)          # note: add_subplot, not plt.subplots -- the second
                                    # panel is 3-D and must be created the same way
ax0.semilogy(np.arange(1, 7), spec_mean, "o-", color=GEO_DARK, zorder=3)
ax0.set_ylim(spec_mean.min() * 0.35, 4.0)
ax0.axvspan(0.6, 2.4, color=GEO_TEAL, alpha=0.12)
ax0.axvspan(2.6, 3.4, color=GEO_RUST, alpha=0.12)
ax0.axvspan(3.6, 6.4, color="0.5", alpha=0.12)
for x, lab in [(1.5, "tangent\nplane"), (3.0, "curv-\nature"), (5.0, "noise\nfloor")]:
    ax0.annotate(lab, xy=(x, 2.0), ha="center", va="center", fontsize=7, color="0.25")
ax0.set_xlabel("local component"); ax0.set_ylabel("mean singular value (normalised)")
ax0.set_title("local spectrum: two large directions\n$\\Rightarrow$ intrinsic dimension 2")

# the same computation on S^2 itself, with tangent planes drawn
sub = X_man[::250]
tree = cKDTree(X_man)
_, nb_idx = tree.query(sub, k=41)
ax = sphere_ax(fig, 122, "local tangent planes on $S^2$", elev=18)
ax.scatter(*X_man[::6].T, s=2, c="0.8", alpha=0.5, lw=0)
for i, p in enumerate(sub):
    nb = X_man[nb_idx[i, 1:]] - p
    _, _, Vl = np.linalg.svd(nb, full_matrices=True)
    e1, e2 = Vl[0] * 0.16, Vl[1] * 0.16
    corners = np.array([p + e1 + e2, p + e1 - e2, p - e1 - e2, p - e1 + e2, p + e1 + e2])
    ax.plot(*corners.T, color=GEO_RUST, lw=0.9)
plt.tight_layout(); plt.show()

> **Exercise 8 — when does the recovery break?**
> (a) Sweep the noise level $\sigma$ from $10^{-4}$ to $1$ and plot the estimated
> intrinsic dimension (say, the number of local singular values above $10\%$ of the
> largest) against $\sigma$. At what noise level does the tangent plane drown?
> Relate the answer to the neighbourhood radius.
>
> (b) Replace the *linear* isometry $A$ by a nonlinear embedding — e.g. append the
> coordinates $(x^2, y^2, z^2, xy, \dots)$ — and repeat. PCA should now fail while
> local PCA still succeeds. Explain the difference in one sentence.
>
> (c) Do the same for $S^3 \subset \mathbb{R}^4$ embedded in $\mathbb{R}^{100}$: does
> local PCA return $3$? How does the required number of neighbours $k$ scale with
> the intrinsic dimension, and how does this reproduce the curse of §6?

**Minha resposta.**

**(a)** Para cada ponto, estimamos a dimensão pelo número de valores singulares locais que satisfazem $s_j>0{,}1s_1$ e tomamos a mediana sobre vários pontos. Fixamos $k=25$: para a amostra sem ruído, esse valor é grande o bastante para amostrar as duas direções tangentes e pequeno o bastante para que a direção de curvatura permaneça abaixo do limiar.

Se $\varepsilon_i,\varepsilon_j\sim\mathcal N(0,\sigma^2I_D)$ são independentes, então

$$\bigl(\mathbb E\|\varepsilon_i-\varepsilon_j\|^2\bigr)^{1/2}=\sigma\sqrt{2D}.$$

Essa é a escala de ruído relevante para as diferenças entre vizinhos. Ela deve ser comparada com o raio $r_k$ da vizinhança. A ruptura não possui um valor universal de $\sigma$: ela ocorre quando a escala de ruído deixa de ser pequena em relação a $r_k$, com a constante determinada pelo limiar espectral adotado.

In [ ]:
def local_dimension_diagnostics(X, k=25, n_query=300, relative_threshold=0.10):
    """Dimensões, espectros relativos e raios de vizinhanças locais."""
    n_query = min(n_query, len(X))
    distances, indices = cKDTree(X).query(X[:n_query], k=k + 1)
    spectra = np.empty((n_query, min(k, X.shape[1])))
    for i in range(n_query):
        patch = X[indices[i, 1:]] - X[i]
        spectra[i] = np.linalg.svd(patch, compute_uv=False)

    relative_spectra = spectra / spectra[:, [0]]
    dimensions = (relative_spectra > relative_threshold).sum(axis=1)
    return {
        "dimensions": dimensions,
        "median_dimension": np.median(dimensions),
        "dimension_quartiles": np.quantile(dimensions, [0.25, 0.75]),
        "median_radius": np.median(distances[:, -1]),
        "mean_relative_spectrum": relative_spectra.mean(axis=0),
    }


rng_ex8_a = np.random.default_rng(SEED + 10)
noise_directions = rng_ex8_a.normal(size=(N_MAN, n_ambient))
clean_linear_image = X_man @ A.T
noise_levels = np.logspace(-4, 0, 17)
k_noise = 25

dimension_statistics = []
neighbourhood_radii = []
for sigma_level in noise_levels:
    noisy_cloud = clean_linear_image + sigma_level * noise_directions
    diagnostic = local_dimension_diagnostics(noisy_cloud, k=k_noise)
    dimension_statistics.append([diagnostic["median_dimension"],
                                 *diagnostic["dimension_quartiles"]])
    neighbourhood_radii.append(diagnostic["median_radius"])

dimension_statistics = np.asarray(dimension_statistics)
median_dimensions = dimension_statistics[:, 0]
lower_quartile, upper_quartile = dimension_statistics[:, 1:].T
neighbourhood_radii = np.asarray(neighbourhood_radii)
clean_radius = local_dimension_diagnostics(clean_linear_image, k=k_noise)["median_radius"]
pairwise_noise_rms = np.sqrt(2 * n_ambient) * noise_levels

break_indices = np.flatnonzero(median_dimensions > 2)
sigma_break = noise_levels[break_indices[0]] if len(break_indices) else np.nan

fig, axes = plt.subplots(1, 2, figsize=(10.2, 3.6))
axes[0].semilogx(noise_levels, median_dimensions, "o-", color=GEO_DARK, ms=4)
axes[0].fill_between(noise_levels, lower_quartile, upper_quartile,
                     color=GEO_TEAL, alpha=0.18, label="intervalo interquartil")
axes[0].axhline(2, color=GEO_RUST, ls="--", label="dimensão verdadeira")
if np.isfinite(sigma_break):
    axes[0].axvline(sigma_break, color="0.4", ls=":")
axes[0].set_xlabel(r"ruído por coordenada $\sigma$")
axes[0].set_ylabel("dimensão local estimada")
axes[0].set_title(r"regra $s_j>0{,}1s_1$ ($k=25$)")
axes[0].legend(fontsize=7)

axes[1].loglog(noise_levels, neighbourhood_radii, "o-", ms=4,
               color=GEO_TEAL, label=r"raio mediano $r_k$")
axes[1].loglog(noise_levels, pairwise_noise_rms, "--", color=GEO_RUST,
               label=r"$\sigma\sqrt{2D}$")
axes[1].axhline(clean_radius, color="0.35", ls=":",
                label=rf"$r_k(0)={clean_radius:.3f}$")
axes[1].set_xlabel(r"ruído por coordenada $\sigma$")
axes[1].set_ylabel("escala de comprimento")
axes[1].set_title("ruído versus tamanho da vizinhança")
axes[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

print(f"raio mediano sem ruído: r_k = {clean_radius:.4f}")
print(f"primeira ruptura da regra dos 10%: sigma = {sigma_break:.4g}")
if np.isfinite(sigma_break):
    relative_noise_at_break = np.sqrt(2 * n_ambient) * sigma_break / clean_radius
    print(f"nessa ruptura, sigma*sqrt(2D)/r_k = {relative_noise_at_break:.3f}")

**(b)** Consideramos a aplicação

$$F(x,y,z)=(x,y,z,x^2,y^2,z^2,xy,xz,yz)\in\mathbb R^9,$$

seguida de uma isometria linear de $\mathbb R^9$ em $\mathbb R^{100}$. Como as três primeiras coordenadas de $F$ são $(x,y,z)$, sua restrição a $S^2$ é injetiva e sua diferencial tem posto $2$; portanto, ela é de fato um mergulho. Depois da centralização, sua imagem possui oito direções lineares relevantes, e não nove, pois $x^2+y^2+z^2=1$ em $S^2$ fornece uma relação afim. Usamos $\sigma=10^{-4}$, pertencente à janela bem resolvida da parte (a), para separar o efeito da não linearidade do efeito do ruído.

**Em uma frase:** PCA global mede o espaço gerado pelas secantes de toda a imagem curva, enquanto PCA local mede a imagem tangente $dF_x(T_xS^2)$, que tem dimensão $2$.

In [ ]:
def quadratic_embedding_s2(X):
    x, y, z = X.T
    return np.column_stack((x, y, z, x**2, y**2, z**2, x*y, x*z, y*z))


rng_ex8_b = np.random.default_rng(SEED + 11)
quadratic_coordinates = quadratic_embedding_s2(X_man)
A_nonlinear = random_isometry(quadratic_coordinates.shape[1], n_ambient, rng_ex8_b)
sigma_nonlinear = 1e-4
X_nonlinear = (quadratic_coordinates @ A_nonlinear.T
               + sigma_nonlinear * rng_ex8_b.normal(size=(N_MAN, n_ambient)))

global_spectrum = np.linalg.svd(X_nonlinear - X_nonlinear.mean(axis=0),
                                compute_uv=False)
global_relative_spectrum = global_spectrum / global_spectrum[0]
nonlinear_local = local_dimension_diagnostics(X_nonlinear, k=12, n_query=400)
local_relative_spectrum = nonlinear_local["mean_relative_spectrum"]

fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.4))
axes[0].semilogy(np.arange(1, 16), global_relative_spectrum[:15], "o-",
                 color=GEO_DARK)
axes[0].axhline(0.1, color=GEO_RUST, ls="--", label="limiar de 10%")
axes[0].set_xlabel("componente global"); axes[0].set_ylabel(r"$s_j/s_1$")
axes[0].set_title("PCA global: oito direções relevantes")
axes[0].legend(fontsize=7)

axes[1].semilogy(np.arange(1, 9), local_relative_spectrum[:8], "o-",
                 color=GEO_TEAL)
axes[1].axhline(0.1, color=GEO_RUST, ls="--", label="limiar de 10%")
axes[1].set_xlabel("componente local"); axes[1].set_ylabel(r"média de $s_j/s_1$")
axes[1].set_title("PCA local: duas direções tangentes")
axes[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

global_dimension = int((global_relative_spectrum > 0.1).sum())
print(f"dimensão linear global pela regra dos 10%: {global_dimension}")
print(f"dimensão local mediana pela mesma regra: "
      f"{nonlinear_local['median_dimension']:.0f}")

**(c)** Para $S^d$, uma vizinhança geodésica pequena de raio $r$ tem volume proporcional a $r^d$. Consequentemente, o raio do $k$-ésimo vizinho satisfaz, em primeira ordem,

$$r_k\asymp\left(\frac{k}{N}\right)^{1/d}.$$

Há duas exigências opostas. Para estimar um espaço tangente $d$-dimensional é necessário ao menos $k\gtrsim d$ (e, para estabilidade estatística, usualmente $k\gg d$); por outro lado, $k$ não pode ser tão grande que $r_k$ torne visível a curvatura. Para conservar um raio máximo $r$ e ainda possuir $k$ vizinhos, é necessário

$$N\gtrsim k r^{-d}.$$

Assim, a quantidade de dados exigida cresce exponencialmente com a dimensão intrínseca quando $r<1$ é fixado: esta é exatamente a maldição da dimensionalidade observada na §6.

In [ ]:
rng_ex8_c = np.random.default_rng(SEED + 12)
sigma_intrinsic = 1e-4
X_s3 = sample_sphere(N_MAN, 4, rng_ex8_c)
A_s3 = random_isometry(4, n_ambient, rng_ex8_c)
X_s3_ambient = (X_s3 @ A_s3.T
                  + sigma_intrinsic * rng_ex8_c.normal(size=(N_MAN, n_ambient)))
X_s2_ambient = (clean_linear_image
                  + sigma_intrinsic * noise_directions)

k_values = np.array([6, 8, 10, 12, 16, 24, 32, 48, 64])
s2_dimensions, s3_dimensions = [], []
s2_radii, s3_radii = [], []
for k in k_values:
    diagnostic_s2 = local_dimension_diagnostics(X_s2_ambient, k=k, n_query=240)
    diagnostic_s3 = local_dimension_diagnostics(X_s3_ambient, k=k, n_query=240)
    s2_dimensions.append(diagnostic_s2["median_dimension"])
    s3_dimensions.append(diagnostic_s3["median_dimension"])
    s2_radii.append(diagnostic_s2["median_radius"])
    s3_radii.append(diagnostic_s3["median_radius"])

s2_dimensions, s3_dimensions = np.asarray(s2_dimensions), np.asarray(s3_dimensions)
s2_radii, s3_radii = np.asarray(s2_radii), np.asarray(s3_radii)
reference_s2 = s2_radii[0] * (k_values / k_values[0])**(1/2)
reference_s3 = s3_radii[0] * (k_values / k_values[0])**(1/3)

fig, axes = plt.subplots(1, 2, figsize=(10.2, 3.6))
axes[0].semilogx(k_values, s2_dimensions, "o-", color=GEO_TEAL, label=r"$S^2$")
axes[0].semilogx(k_values, s3_dimensions, "s-", color=GEO_RUST, label=r"$S^3$")
axes[0].axhline(2, color=GEO_TEAL, ls=":")
axes[0].axhline(3, color=GEO_RUST, ls=":")
axes[0].set_xlabel(r"número de vizinhos $k$")
axes[0].set_ylabel("dimensão local mediana")
axes[0].set_title("janela de escalas da PCA local")
axes[0].legend()

axes[1].loglog(k_values, s2_radii, "o-", color=GEO_TEAL, label=r"$S^2$: medido")
axes[1].loglog(k_values, reference_s2, "--", color=GEO_TEAL,
               label=r"referência $k^{1/2}$")
axes[1].loglog(k_values, s3_radii, "s-", color=GEO_RUST, label=r"$S^3$: medido")
axes[1].loglog(k_values, reference_s3, "--", color=GEO_RUST,
               label=r"referência $k^{1/3}$")
axes[1].set_xlabel(r"número de vizinhos $k$")
axes[1].set_ylabel(r"raio mediano $r_k$")
axes[1].set_title("vizinhanças são maiores em dimensão maior")
axes[1].legend(fontsize=7, ncol=2)
plt.tight_layout(); plt.show()

successful_k_s3 = k_values[s3_dimensions == 3]
print("dimensões medianas estimadas para S^3:")
print(dict(zip(k_values.tolist(), s3_dimensions.astype(int).tolist())))
print("valores de k para os quais a regra retorna 3:", successful_k_s3.tolist())

---
## 8. Supervised learning on $S^2$ as energy minimisation

Now we assemble Lecture 2's definition, with every ingredient made explicit.

| Symbol | Here |
|---|---|
| $(X, \mu)$ | $S^2$ with the uniform measure |
| target $f^\star$ | a fixed harmonic polynomial, restricted to $S^2$ |
| labels | $y_i = f^\star(x_i) + \varepsilon_i$, $\varepsilon_i \sim \mathcal{N}(0,\sigma^2)$ |
| $\mathcal{H}$ | polynomials in $(x,y,z)$ of degree $\leq D$, restricted to $S^2$ |
| $\ell$ | squared error |
| algorithm | exact least squares (the convex world of Lecture 3) |

### 8.1 The target

We take a combination of solid harmonics — polynomials with $\Delta_{\mathbb{R}^3} p = 0$,
whose restrictions to $S^2$ are eigenfunctions of the Laplace–Beltrami operator,
$\Delta_{S^2} Y_\ell = -\ell(\ell+1) Y_\ell$. This is the spectral decomposition the
syllabus promises and Tutorial 11 will build numerically.

In [ ]:
def Y2(X):   # degree 2, harmonic: Delta(2z^2 - x^2 - y^2) = 4 - 2 - 2 = 0
    x, y, z = X.T
    return 2 * z**2 - x**2 - y**2


def Y2b(X):  # degree 2, harmonic
    x, y, z = X.T
    return x * y


def Y3(X):   # degree 3, harmonic: Re((x+iy)^3)
    x, y, z = X.T
    return x**3 - 3 * x * y**2


def f_star(X):
    return 0.6 * Y2(X) + 1.1 * Y2b(X) + 0.9 * Y3(X)


# Monte-Carlo check: harmonics of different degree are L^2(S^2)-orthogonal
big = sample_sphere(200000, 3, rng)
print("<Y2, Y3>  =", (Y2(big) * Y3(big)).mean().round(4), " (should be ~0)")
print("<Y2, Y2b> =", (Y2(big) * Y2b(big)).mean().round(4), " (should be ~0)")
print("<Y2, 1>   =", Y2(big).mean().round(4), "  (harmonics have zero mean)")

In [ ]:
X_vis = fibonacci_sphere(6000)
f_vis = f_star(X_vis)

fig = plt.figure(figsize=(11.2, 4.0))
ax1 = sphere_ax(fig, 131, "$f^\\star$ on $S^2$", elev=22)
sc = ax1.scatter(*X_vis.T, c=f_vis, cmap="coolwarm", s=6, lw=0)
ax2 = sphere_ax(fig, 132, "the far side", elev=22, azim=215)
ax2.scatter(*X_vis.T, c=f_vis, cmap="coolwarm", s=6, lw=0)

ax3 = fig.add_subplot(133, projection="mollweide")
lon = np.arctan2(X_vis[:, 1], X_vis[:, 0])
lat = np.arcsin(np.clip(X_vis[:, 2], -1, 1))
ax3.scatter(lon, lat, c=f_vis, cmap="coolwarm", s=3, lw=0)
ax3.set_title("equal-area (Mollweide) projection", pad=12)
ax3.grid(alpha=0.25); ax3.set_xticklabels([]); ax3.tick_params(labelsize=6)
ax3.set_yticks(np.deg2rad([-60, -30, 0, 30, 60]))
fig.colorbar(sc, ax=ax3, shrink=0.62, pad=0.06, label="$f^\\star$")
plt.tight_layout(); plt.show()

### 8.2 The hypothesis space, and a subtlety geometers will enjoy

Take $\mathcal{H}_D$ = restrictions to $S^2$ of polynomials of degree $\leq D$ in
three variables. The obvious basis is the monomials, of which there are
$\binom{D+3}{3}$ — but the restriction map

$$\mathbb{R}[x,y,z]_{\leq D} \longrightarrow C^\infty(S^2)$$

has a kernel: the ideal generated by $x^2+y^2+z^2-1$. So the design matrix is
**rank deficient by construction**, and any solver must handle that. We use
`np.linalg.lstsq`, which returns the minimum-norm least-squares solution via the
SVD — a choice with real consequences that we come back to at the end.

In [ ]:
from itertools import combinations_with_replacement


def poly_features(X, degree):
    """All monomials of degree <= `degree` in the columns of X."""
    n, d = X.shape
    cols = [np.ones(n)]
    for deg in range(1, degree + 1):
        for combo in combinations_with_replacement(range(d), deg):
            cols.append(np.prod(X[:, combo], axis=1))
    return np.stack(cols, axis=1)


for D in range(1, 9):
    Phi = poly_features(X_gauss[:500], D)
    print(f"degree {D:2d}:  {Phi.shape[1]:4d} monomials, "
          f"rank {np.linalg.matrix_rank(Phi):4d}   "
          f"(dim of the space of restrictions = {(D+1)**2})")

The rank is exactly $(D+1)^2 = \sum_{\ell=0}^{D}(2\ell+1)$: the dimension of the
space of spherical harmonics of degree $\leq D$. The numerics have just recovered
the decomposition $L^2(S^2) = \bigoplus_{\ell \geq 0} \mathcal{H}_\ell$ of the
Laplacian's eigenspaces. Nothing about this was put in by hand.

### 8.3 Train / validation / test

Lecture 2's universal protocol. Note that we can afford an enormous test set here
because we can *generate* data — the situation the lecture called "mathematics is
rich in data, cheap to sample at scale, and noiseless". Real practitioners cannot
do this, and their error bars suffer for it.

In [ ]:
NOISE = 0.15
N_TRAIN, N_VAL, N_TEST = 80, 200, 20000

X_tr = sample_sphere(N_TRAIN, 3, rng)
X_va = sample_sphere(N_VAL, 3, rng)
X_te = sample_sphere(N_TEST, 3, rng)

y_tr = f_star(X_tr) + NOISE * rng.normal(size=N_TRAIN)
y_va = f_star(X_va) + NOISE * rng.normal(size=N_VAL)
y_te = f_star(X_te)                       # noiseless: we measure approximation error

baseline = np.mean((y_te - y_tr.mean()) ** 2)
print(f"trivial baseline (predict the training mean): MSE = {baseline:.4f}")

In [ ]:
def fit_lstsq(X_tr, y_tr, degree, ridge=0.0):
    """Empirical risk minimisation over polynomials of the given degree."""
    Phi = poly_features(X_tr, degree)
    if ridge > 0:
        A = np.vstack([Phi, np.sqrt(ridge) * np.eye(Phi.shape[1])])
        b = np.concatenate([y_tr, np.zeros(Phi.shape[1])])
        w, *_ = np.linalg.lstsq(A, b, rcond=None)
    else:
        w, *_ = np.linalg.lstsq(Phi, y_tr, rcond=None)
    return w


def mse(X, y, w, degree):
    return np.mean((poly_features(X, degree) @ w - y) ** 2)


degrees = np.arange(1, 13)
tr_err, va_err, te_err = [], [], []
for D in degrees:
    w = fit_lstsq(X_tr, y_tr, D)
    tr_err.append(mse(X_tr, y_tr, w, D))
    va_err.append(mse(X_va, y_va, w, D))
    te_err.append(mse(X_te, y_te, w, D))

D_best = degrees[int(np.argmin(va_err))]
print(f"validation selects degree {D_best};  "
      f"its test MSE = {te_err[int(np.argmin(va_err))]:.5f}  "
      f"(baseline {baseline:.4f})")

### 8.4 The U-curve, on real numbers

Here is Lecture 2's slide-7 sketch, reproduced as an experiment. Training error
falls monotonically — a larger $\mathcal{H}$ can only fit the sample better, and by
degree $8$ it has reached $10^{-27}$, which is to say exactly zero. Test error
falls, bottoms out at $D = 3$, the true degree of $f^\star$, then explodes.

In [ ]:
n_mono = [(D + 1) * (D + 2) * (D + 3) // 6 for D in degrees]   # monomials written down
n_eff = [(D + 1) ** 2 for D in degrees]                        # ... of which independent

fig, axes = plt.subplots(1, 2, figsize=(10.0, 3.4))

ax = axes[0]
ax.semilogy(degrees, tr_err, "o-", color=GEO_DARK, label="train")
ax.semilogy(degrees, va_err, "s-", color=GEO_TEAL, label="validation")
ax.semilogy(degrees, te_err, "^-", color=GEO_RUST, label="test (noiseless)")
ax.axhline(baseline, color="0.4", ls=":", lw=1.4, label="trivial baseline")
ax.axhline(NOISE**2, color="0.4", ls="--", lw=1.2, label=r"noise floor $\sigma^2$")
ax.axvline(3, color="0.75", lw=6, alpha=0.35, zorder=0)
ax.set_ylim(3e-4, 3e3)          # the train error dives to 1e-27; clip so the U is visible
ax.annotate("true degree", xy=(3, 8e-4), fontsize=7, ha="center", color="0.35")
ax.annotate("train error $\\to 10^{-27}$:\nexact interpolation", xy=(9.6, 6e-4),
            fontsize=6.5, ha="center", color=GEO_DARK)
ax.set_xlabel("polynomial degree $D$   (capacity of $\\mathcal{H}$)")
ax.set_ylabel("mean squared error")
ax.set_title(f"the classical U-curve, $N_{{train}} = {N_TRAIN}$")
ax.legend(fontsize=6.5, loc="upper left", ncol=2)

ax = axes[1]
ax.loglog(n_eff, te_err, "^-", color=GEO_RUST, label="test MSE")
ax.axvline(N_TRAIN, color=GEO_DARK, ls="--", lw=1.3)
ax.annotate("$p = N$: interpolation threshold", xy=(N_TRAIN * 0.93, 2e-2),
            fontsize=7, rotation=90, ha="right", va="bottom", color=GEO_DARK)
ax.set_xlabel(r"effective parameters $p = (D+1)^2$"); ax.set_ylabel("test MSE")
ax.set_title("the same data against effective capacity")
for D, x, y in zip(degrees, n_eff, te_err):
    if D in (3, 8, 12):
        ax.annotate(f"$D={D}$", xy=(x, y), xytext=(3, 5), textcoords="offset points",
                    fontsize=7, color=GEO_DARK)
plt.tight_layout(); plt.show()

Two things in the right-hand panel deserve a moment.

**The horizontal axis is $(D+1)^2$, not the number of monomials.** §8.2 showed the
design matrix has rank $(D+1)^2$ regardless of how many monomials we write down, so
that is the honest count of parameters. Choosing the wrong notion of "capacity"
would have put the peak in the wrong place.

**The peak sits exactly at $p = N$.** With $N_{\text{train}} = 80$, the blow-up
occurs at $D = 8$, where $(D+1)^2 = 81$: the first model with just enough freedom to
interpolate all $80$ training points and none left over. This is the
**interpolation threshold**, and the catastrophe there is real — the test error is
some four orders of magnitude worse than at $D = 3$.

And then, past the threshold, **the test error comes back down** — by an order of
magnitude, though it does not recover anything like the quality of $D=3$. That is
Lecture 2's promised "modern twist", **double descent**, appearing in a model with
no neural network anywhere in sight. The mechanism is visible in our code: once
$p > N$ there are infinitely many interpolating solutions, and `np.linalg.lstsq`
silently picks the one of *minimum norm*. That choice is an implicit regulariser,
and it improves as $p$ grows. Lecture 4 returns to this.

Notice also, in the printed output above, that **validation and test disagree
slightly** about the best degree: the test minimum is at $D=3$, the true degree of
$f^\star$, while the validation set — a mere $200$ noisy points — often prefers
$D=4$. Nothing has gone wrong. Model selection on a finite validation set is itself
a noisy estimate, and the difference between the two candidates is well inside that
noise. It is a small illustration of why the protocol has three splits and not two:
had we selected *and* reported on the same set, we would have quoted an
optimistically biased number.

### 8.5 Regularisation

The other lever from slide 7: keep the large $\mathcal{H}$, but penalise complexity.
Adding $\lambda\|w\|^2$ to the empirical risk — Tikhonov, exactly as in ill-posed
inverse problems — tames the high-degree models completely.

In [ ]:
lams = np.logspace(-8, 1, 40)
D_big = 12
curve_tr, curve_te = [], []
for lam in lams:
    w = fit_lstsq(X_tr, y_tr, D_big, ridge=lam)
    curve_tr.append(mse(X_tr, y_tr, w, D_big))
    curve_te.append(mse(X_te, y_te, w, D_big))

fig, ax = plt.subplots(figsize=(5.6, 3.2))
ax.loglog(lams, curve_tr, "o-", color=GEO_DARK, ms=3, label="train")
ax.loglog(lams, curve_te, "^-", color=GEO_RUST, ms=3, label="test")
ax.axhline(min(te_err), color=GEO_TEAL, ls="--", lw=1.2,
           label="best unregularised degree")
ax.set_xlabel(r"ridge parameter $\lambda$"); ax.set_ylabel("MSE")
ax.set_title(f"degree {D_big}: {n_mono[-1]} monomials / {n_eff[-1]} effective\nparameters, against {N_TRAIN} training points")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

print(f"best test MSE with regularisation: {min(curve_te):.5f} "
      f"at lambda = {lams[int(np.argmin(curve_te))]:.2e}")

### 8.6 Look at the errors, not only at the number

A scalar loss hides *where* a model fails. Plot the prediction and the residual on
the sphere: the residual should be structureless if the model has captured
everything, and should show the missing harmonic if it has not.

In [ ]:
w_good = fit_lstsq(X_tr, y_tr, D_best)
w_over = fit_lstsq(X_tr, y_tr, 11)

pred_good = poly_features(X_vis, D_best) @ w_good
pred_over = poly_features(X_vis, 11) @ w_over
truth = f_star(X_vis)

fig = plt.figure(figsize=(11.0, 6.4))
panels = [
    ("truth $f^\\star$", truth, "coolwarm", None),
    (f"fit, degree {D_best}", pred_good, "coolwarm", None),
    ("fit, degree 11 (overfit)", pred_over, "coolwarm", None),
    ("", None, None, None),
    (f"residual, degree {D_best}", pred_good - truth, "PuOr", 0.6),
    ("residual, degree 11", pred_over - truth, "PuOr", 0.6),
]
for k, (title, vals, cmap, vlim) in enumerate(panels):
    if vals is None:
        continue
    ax = sphere_ax(fig, 231 + k, title, elev=22)
    kw = dict(vmin=-vlim, vmax=vlim) if vlim else {}
    s = ax.scatter(*X_vis.T, c=vals, cmap=cmap, s=5, lw=0, **kw)
    fig.colorbar(s, ax=ax, shrink=0.55)
# training points, to show how few there are
ax = sphere_ax(fig, 234, f"the {N_TRAIN} training points", elev=22)
draw_wire_sphere(ax, alpha=0.12)
ax.scatter(*X_tr.T, c=y_tr, cmap="coolwarm", s=34, depthshade=False, edgecolor="k", lw=0.3)
plt.tight_layout(); plt.show()

> **Exercise 9 — the learning curve.**
> Fix the degree at the value validation selected, and sweep $N_{\text{train}}$ from
> $10$ to $5000$ on log axes. Plot test MSE against $N$, averaged over (say) 20
> random draws so that you see the *variance* as well as the mean. You should find
> the error falling like a power of $N$ until it saturates at the noise floor
> $\sigma^2$. Which of the three terms in the bias–variance decomposition does each
> regime correspond to?

**Minha resposta.** 

Seja $\widehat f_N$ o estimador obtido de uma amostra de treinamento aleatória com $N$ pontos. Para um novo rótulo $Y=f^\star(X)+\varepsilon$, com $\mathbb E[\varepsilon\mid X]=0$ e $\operatorname{Var}(\varepsilon\mid X)=\sigma^2$, vale a decomposição

$$\mathbb E\bigl[(Y-\widehat f_N(X))^2\bigr]
=\underbrace{\sigma^2}_{\text{ruído irredutível}}
+\underbrace{\|\mathbb E\widehat f_N-f^\star\|_{L^2(S^2)}^2}_{\text{viés}^2}
+\underbrace{\mathbb E\|\widehat f_N-\mathbb E\widehat f_N\|_{L^2(S^2)}^2}_{\text{variância}}.$$

O conjunto $y_{\rm te}$ da §8.3 é deliberadamente **sem ruído**; logo, seu MSE estima apenas a soma $\text{viés}^2+\text{variância}$ e deve tender a zero. Para observar o piso $\sigma^2$ pedido no exercício, somamos esse termo irredutível, o que equivale ao risco esperado contra um novo rótulo ruidoso. Como $D_{\rm best}\geq3$ neste experimento e $f^\star\in\mathcal H_3$, o viés de aproximação é nulo no limite; depois do regime de interpolação, a variância decai aproximadamente como $p/N$, até que o risco total fique visualmente indistinguível de $\sigma^2$.

In [ ]:
rng_ex9 = np.random.default_rng(SEED + 13)
learning_sizes = np.unique(np.rint(np.geomspace(10, 5000, 13)).astype(int))
n_learning_repeats = 20
n_learning_test = 6000
degree_learning = int(D_best)

X_learning_test = sample_sphere(n_learning_test, 3, rng_ex9)
truth_learning_test = f_star(X_learning_test)
Phi_learning_test = poly_features(X_learning_test, degree_learning)
learning_predictions = np.empty(
    (n_learning_repeats, len(learning_sizes), n_learning_test)
)

for repeat in range(n_learning_repeats):
    X_pool = sample_sphere(int(learning_sizes[-1]), 3, rng_ex9)
    y_pool = f_star(X_pool) + NOISE * rng_ex9.normal(size=len(X_pool))
    Phi_pool = poly_features(X_pool, degree_learning)
    for j, n_train in enumerate(learning_sizes):
        weights, *_ = np.linalg.lstsq(
            Phi_pool[:n_train], y_pool[:n_train], rcond=None
        )
        learning_predictions[repeat, j] = Phi_learning_test @ weights

mean_prediction = learning_predictions.mean(axis=0)
squared_bias = np.mean(
    (mean_prediction - truth_learning_test[None, :])**2, axis=1
)
estimator_variance = np.mean(
    (learning_predictions - mean_prediction[None, :, :])**2, axis=(0, 2)
)
signal_risk_by_repeat = np.mean(
    (learning_predictions - truth_learning_test[None, None, :])**2, axis=2
)
signal_risk = signal_risk_by_repeat.mean(axis=0)
observed_risk = signal_risk + NOISE**2
risk_q10, risk_q90 = np.quantile(
    signal_risk_by_repeat + NOISE**2, [0.10, 0.90], axis=0
)

p_learning = (degree_learning + 1)**2
power_window = (learning_sizes >= 4 * p_learning) & (learning_sizes <= 2000)
variance_power = np.polyfit(
    np.log(learning_sizes[power_window]),
    np.log(estimator_variance[power_window]), 1
)[0]

fig, axes = plt.subplots(1, 2, figsize=(10.2, 3.6))
axes[0].loglog(learning_sizes, observed_risk, "o-", color=GEO_DARK,
               label=r"risco para novo $Y$")
axes[0].fill_between(learning_sizes, risk_q10, risk_q90,
                     color=GEO_DARK, alpha=0.14, label="quantis 10%--90%")
axes[0].loglog(learning_sizes, signal_risk, "s-", color=GEO_TEAL,
               label=r"erro contra $f^\star$")
axes[0].axhline(NOISE**2, color=GEO_RUST, ls="--",
                label=r"piso $\sigma^2$")
axes[0].axvline(p_learning, color="0.5", ls=":",
                label=rf"$p=(D+1)^2={p_learning}$")
axes[0].set_xlabel(r"amostras de treinamento $N$")
axes[0].set_ylabel("erro quadrático médio")
axes[0].set_title(rf"curva de aprendizagem, $D={degree_learning}$")
axes[0].legend(fontsize=7)

axes[1].loglog(learning_sizes, squared_bias, "o-", color=GEO_RUST,
               label=r"viés$^2$")
axes[1].loglog(learning_sizes, estimator_variance, "s-", color=GEO_TEAL,
               label="variância")
axes[1].loglog(learning_sizes, squared_bias + estimator_variance, "^-",
               color=GEO_DARK, label=r"viés$^2+$variância")
axes[1].axhline(NOISE**2, color="0.4", ls="--", label=r"ruído $\sigma^2$")
axes[1].set_xlabel(r"amostras de treinamento $N$")
axes[1].set_ylabel("contribuição ao risco")
axes[1].set_title("decomposição empírica")
axes[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

decomposition_error = np.max(
    np.abs(signal_risk - squared_bias - estimator_variance)
)
print(f"D fixado pela validação: {degree_learning} (p={p_learning})")
print(f"potência medida para a variância: N^{variance_power:.3f}")
print(f"erro numérico máximo na identidade viés-variância: "
      f"{decomposition_error:.2e}")
print(f"risco final para novo rótulo: {observed_risk[-1]:.5f}; "
      f"piso sigma^2 = {NOISE**2:.5f}")

> **Exercise 10 — equivariance, cheaply.**
> The target $f^\star$ was built from harmonics, and the space $\mathcal{H}_\ell$ is an
> irreducible representation of $SO(3)$. Exploit this: instead of raw monomials, fit
> in a basis of *harmonic* polynomials of degree exactly $\ell$ for $\ell \leq D$
> (obtain it by orthonormalising the monomial features against a large uniform
> sample — a QR factorisation). Compare the conditioning of the two design matrices
> and the test error at fixed $N_{\text{train}}$.
>
> This is the whole idea of Lecture 10 in miniature: **the same hypothesis space in a
> symmetry-adapted basis is a better hypothesis space in practice.**

**Minha resposta.** 

Denote por $\mathcal P_{\leq \ell}|_{S^2}$ o espaço das restrições de polinômios de grau no máximo $\ell$. A decomposição harmônica pode ser caracterizada por

$$\mathcal H_\ell=\mathcal P_{\leq \ell}|_{S^2}\ominus
\mathcal P_{\leq \ell-1}|_{S^2},
\qquad \dim\mathcal H_\ell=2\ell+1.$$

Em uma grande amostra uniforme de referência, projetamos os monômios homogêneos de grau $\ell$ no complemento ortogonal dos blocos de graus anteriores e aplicamos uma fatoração ortogonal (a SVD desempenha aqui o mesmo papel numérico da QR com detecção de posto). Isso produz blocos empiricamente ortonormais de tamanhos $1,3,5,\ldots,2D+1$. Sob uma rotação, cada bloco $\mathcal H_\ell$ é preservado e suas coordenadas mudam por uma matriz ortogonal; por isso a base respeita a ação de $SO(3)$.

As bases monomial e harmônica geram exatamente o mesmo espaço de funções. Em aritmética exata e no regime sobre-determinado, mínimos quadrados produz a mesma função nas duas bases; a vantagem da base harmônica é remover relações nulas, melhorar o condicionamento e tornar penalizações por norma geometricamente significativas.

In [ ]:
def empirical_harmonic_transform(X_reference, degree):
    """Transforma monômios em blocos L2-ortogonais que aproximam H_ell."""
    Phi_reference = poly_features(X_reference, degree)
    n_reference = len(X_reference)
    Q_previous = np.empty((n_reference, 0))
    block_dimensions = []

    for ell in range(degree + 1):
        # Número de monômios de graus < ell e <= ell, respectivamente.
        start = ell * (ell + 1) * (ell + 2) // 6
        stop = (ell + 1) * (ell + 2) * (ell + 3) // 6
        homogeneous = Phi_reference[:, start:stop]
        residual = homogeneous - Q_previous @ (Q_previous.T @ homogeneous)
        U, singular_values, _ = np.linalg.svd(residual, full_matrices=False)
        block_dimension = 2 * ell + 1
        if singular_values[block_dimension - 1] <= 1e-10 * singular_values[0]:
            raise RuntimeError("A amostra de referência não resolveu o bloco harmônico.")
        Q_block = U[:, :block_dimension]
        # Uma segunda ortogonalização remove erros acumulados de arredondamento.
        Q_block -= Q_previous @ (Q_previous.T @ Q_block)
        Q_block, _ = np.linalg.qr(Q_block)
        Q_previous = np.column_stack((Q_previous, Q_block))
        block_dimensions.append(block_dimension)

    # sqrt(n) converte ortonormalidade euclidiana em ortonormalidade para
    # o produto interno empírico n^{-1} sum_i f(x_i)g(x_i).
    target_values = np.sqrt(n_reference) * Q_previous
    transform, *_ = np.linalg.lstsq(Phi_reference, target_values, rcond=None)
    return transform, block_dimensions


rng_ex10 = np.random.default_rng(SEED + 14)
D_harmonic = 7
X_harmonic_reference = sample_sphere(12000, 3, rng_ex10)
harmonic_transform, harmonic_block_dimensions = empirical_harmonic_transform(
    X_harmonic_reference, D_harmonic
)

Phi_raw_train = poly_features(X_tr, D_harmonic)
Phi_raw_test = poly_features(X_te, D_harmonic)
Phi_harmonic_train = Phi_raw_train @ harmonic_transform
Phi_harmonic_test = Phi_raw_test @ harmonic_transform

raw_singular_values = np.linalg.svd(Phi_raw_train, compute_uv=False)
harmonic_singular_values = np.linalg.svd(Phi_harmonic_train, compute_uv=False)
raw_rank = np.linalg.matrix_rank(Phi_raw_train)
harmonic_rank = np.linalg.matrix_rank(Phi_harmonic_train)
raw_effective_condition = raw_singular_values[0] / raw_singular_values[raw_rank - 1]
harmonic_condition = harmonic_singular_values[0] / harmonic_singular_values[-1]

weights_raw, *_ = np.linalg.lstsq(Phi_raw_train, y_tr, rcond=None)
weights_harmonic, *_ = np.linalg.lstsq(Phi_harmonic_train, y_tr, rcond=None)
prediction_raw = Phi_raw_test @ weights_raw
prediction_harmonic = Phi_harmonic_test @ weights_harmonic
error_raw_basis = np.mean((prediction_raw - y_te)**2)
error_harmonic_basis = np.mean((prediction_harmonic - y_te)**2)

Phi_harmonic_reference = (
    poly_features(X_harmonic_reference, D_harmonic) @ harmonic_transform
)
reference_gram = Phi_harmonic_reference.T @ Phi_harmonic_reference / len(X_harmonic_reference)
orthonormality_error = np.linalg.norm(
    reference_gram - np.eye(reference_gram.shape[0]), ord=2
)

fig, axes = plt.subplots(1, 2, figsize=(10.0, 3.4))
axes[0].semilogy(
    np.arange(1, len(raw_singular_values) + 1),
    np.maximum(raw_singular_values / raw_singular_values[0], 1e-18),
    "o-", ms=3, color=GEO_RUST, label="monômios redundantes"
)
axes[0].semilogy(
    np.arange(1, len(harmonic_singular_values) + 1),
    harmonic_singular_values / harmonic_singular_values[0],
    "s-", ms=3, color=GEO_TEAL, label="base harmônica"
)
axes[0].set_xlabel("índice singular"); axes[0].set_ylabel(r"$s_j/s_1$")
axes[0].set_title("condicionamento da matriz de design")
axes[0].legend(fontsize=7)

axes[1].semilogy(
    [0, 1], [error_raw_basis, error_harmonic_basis], "o", ms=8,
    color=GEO_DARK
)
axes[1].set_xticks([0, 1], ["monomial", "harmônica"])
axes[1].set_xlim(-0.5, 1.5); axes[1].set_ylabel("MSE de teste sem ruído")
common_basis_error = max(error_raw_basis, error_harmonic_basis)
axes[1].set_ylim(common_basis_error / 2, common_basis_error * 2)
axes[1].set_title(rf"mesmo $\mathcal{{H}}_{{{D_harmonic}}}$, mesmo ajuste")
plt.tight_layout(); plt.show()

print("dimensões dos blocos H_ell:", harmonic_block_dimensions)
print(f"monômios: {Phi_raw_train.shape[1]} colunas, posto {raw_rank}; "
      f"condicionamento efetivo {raw_effective_condition:.2e}")
print(f"harmônicos: {Phi_harmonic_train.shape[1]} colunas, posto {harmonic_rank}; "
      f"condicionamento {harmonic_condition:.2e}")
print(f"erro de ortonormalidade na referência: {orthonormality_error:.2e}")
print(f"MSE monomial = {error_raw_basis:.6f}; "
      f"MSE harmônico = {error_harmonic_basis:.6f}")
print(f"máxima diferença entre as previsões: "
      f"{np.max(np.abs(prediction_raw - prediction_harmonic)):.2e}")

> **Exercise 11 — a real geometric label.** *(mini-project seed)*
> Replace $f^\star$ by something you have to *compute* rather than write down. For
> instance: sample an ellipsoid $\{x^2/a^2 + y^2/b^2 + z^2/c^2 = 1\}$, and let the
> label at each point be the Gaussian curvature there (closed form available, so you
> can check). Train a model to predict curvature from the local point-cloud
> geometry alone — say, from the sorted distances to the $k$ nearest neighbours,
> which is a rotation- and translation-invariant feature.
>
> Does it transfer to a surface it was never trained on? This question — *does a
> geometric quantity learned on one family of shapes generalise to another* — is a
> perfectly good starting point for the mini-project.

**Minha resposta.** 

Para o elipsoide

$$E_{a,b,c}=\left\{(x,y,z):\frac{x^2}{a^2}+\frac{y^2}{b^2}+\frac{z^2}{c^2}=1\right\},$$

a curvatura gaussiana é

$$K(x,y,z)=\frac{1}{a^2b^2c^2
\left(x^2/a^4+y^2/b^4+z^2/c^4\right)^2}.$$

Para cada ponto $p_i$ usamos como dados geométricos os logaritmos de distâncias selecionadas da lista ordenada

$$d_{i,(1)}\leq\cdots\leq d_{i,(k)}.$$

Essas características são invariantes por movimentos rígidos, pois $\|Rp_i+t-(Rp_j+t)\|=\|p_i-p_j\|$ para $R\in O(3)$. Elas não são invariantes por mudança de escala; por isso usamos elipsoides de volume proporcional constante, $abc=1$, e prevemos $\log K$. O treinamento, a validação e o teste são separados por elipsoide inteiro. Assim, o erro de teste mede transferência entre superfícies, e não interpolação de novos pontos de uma superfície já observada.

Invariância não implica suficiência: a lista de distâncias ao ponto central descarta a disposição angular dos vizinhos, enquanto $K=\det S_p$ depende das duas direções principais do operador de forma. Portanto, um resultado negativo indicará que essa representação local é pobre, e não que a curvatura deixe de ser uma quantidade geometricamente aprendível.

In [ ]:
def sample_ellipsoid_surface(n_points, axes, rng):
    """Amostra aproximadamente uniforme para a medida de área do elipsoide."""
    axes = np.asarray(axes, dtype=float)
    accepted = []
    n_accepted = 0
    # Para L=diag(a,b,c), o jacobiano de área é
    # det(L) ||L^{-T}u||; seu máximo ocorre no menor semieixo.
    maximum_weight = np.prod(axes) / axes.min()
    while n_accepted < n_points:
        batch_size = max(2000, 2 * (n_points - n_accepted))
        directions = sample_sphere(batch_size, 3, rng)
        weights = np.prod(axes) * np.linalg.norm(directions / axes, axis=1)
        keep = rng.random(batch_size) < weights / maximum_weight
        accepted.append(directions[keep])
        n_accepted += keep.sum()
    return np.vstack(accepted)[:n_points] * axes


def ellipsoid_gaussian_curvature(points, axes):
    axes = np.asarray(axes, dtype=float)
    denominator = np.sum(points**2 / axes**4, axis=1)**2
    return 1.0 / (np.prod(axes**2) * denominator)


def ellipsoid_point_cloud_dataset(axes, n_points, k, feature_positions, rng):
    points = sample_ellipsoid_surface(n_points, axes, rng)
    distances, _ = cKDTree(points).query(points, k=k + 1)
    sorted_neighbour_distances = distances[:, 1:]
    # Poucos quantis logarítmicos são mais estáveis que todas as 40 distâncias.
    base_features = np.log(sorted_neighbour_distances[:, feature_positions])
    log_curvature = np.log(ellipsoid_gaussian_curvature(points, axes))
    return points, base_features, log_curvature


def fit_matrix_ridge(Phi, targets, regularisation):
    penalty = np.eye(Phi.shape[1])
    penalty[0, 0] = 0.0  # não penalizamos o termo constante
    normal_matrix = Phi.T @ Phi / len(Phi) + regularisation * penalty
    normal_vector = Phi.T @ targets / len(Phi)
    return np.linalg.solve(normal_matrix, normal_vector)


rng_ex11 = np.random.default_rng(SEED + 15)
n_ellipsoid_points = 2500
k_ellipsoid = 40
feature_positions = np.unique(
    np.rint(np.geomspace(1, k_ellipsoid, 10)).astype(int)
) - 1

training_axes = [
    (1.00, 1.00, 1.00),
    (1.20, 1.00, 1 / 1.20),
    (1.35, 1.10, 1 / (1.35 * 1.10)),
    (1.45, 0.90, 1 / (1.45 * 0.90)),
]
validation_axes = (1.30, 0.85, 1 / (1.30 * 0.85))
test_axes = (1.60, 1.05, 1 / (1.60 * 1.05))

training_datasets = [
    ellipsoid_point_cloud_dataset(
        axes, n_ellipsoid_points, k_ellipsoid, feature_positions, rng_ex11
    )
    for axes in training_axes
]
X_ellipsoid_train_base = np.vstack([dataset[1] for dataset in training_datasets])
y_ellipsoid_train = np.concatenate([dataset[2] for dataset in training_datasets])

points_ellipsoid_val, X_ellipsoid_val_base, y_ellipsoid_val = (
    ellipsoid_point_cloud_dataset(
        validation_axes, n_ellipsoid_points, k_ellipsoid, feature_positions, rng_ex11
    )
)
points_ellipsoid_test, X_ellipsoid_test_base, y_ellipsoid_test = (
    ellipsoid_point_cloud_dataset(
        test_axes, n_ellipsoid_points, k_ellipsoid, feature_positions, rng_ex11
    )
)

ellipsoid_feature_mean = X_ellipsoid_train_base.mean(axis=0)
ellipsoid_feature_std = X_ellipsoid_train_base.std(axis=0)
standardise_ellipsoid = lambda features: (
    (features - ellipsoid_feature_mean) / ellipsoid_feature_std
)
Phi_ellipsoid_train = poly_features(
    standardise_ellipsoid(X_ellipsoid_train_base), degree=2
)
Phi_ellipsoid_val = poly_features(
    standardise_ellipsoid(X_ellipsoid_val_base), degree=2
)
Phi_ellipsoid_test = poly_features(
    standardise_ellipsoid(X_ellipsoid_test_base), degree=2
)

ellipsoid_ridges = np.logspace(-6, 4, 32)
ellipsoid_validation_errors = []
for regularisation in ellipsoid_ridges:
    coefficients = fit_matrix_ridge(
        Phi_ellipsoid_train, y_ellipsoid_train, regularisation
    )
    ellipsoid_validation_errors.append(
        np.mean((Phi_ellipsoid_val @ coefficients - y_ellipsoid_val)**2)
    )
best_ellipsoid_ridge = ellipsoid_ridges[np.argmin(ellipsoid_validation_errors)]
ellipsoid_coefficients = fit_matrix_ridge(
    Phi_ellipsoid_train, y_ellipsoid_train, best_ellipsoid_ridge
)
predicted_log_curvature = Phi_ellipsoid_test @ ellipsoid_coefficients
true_curvature_test = np.exp(y_ellipsoid_test)
predicted_curvature_test = np.exp(predicted_log_curvature)
relative_curvature_error = (
    predicted_curvature_test - true_curvature_test
) / true_curvature_test

test_log_mse = np.mean((predicted_log_curvature - y_ellipsoid_test)**2)
baseline_log_mse = np.mean((y_ellipsoid_train.mean() - y_ellipsoid_test)**2)
test_log_r2 = 1 - test_log_mse / np.var(y_ellipsoid_test)
median_relative_error = np.median(np.abs(relative_curvature_error))

fig = plt.figure(figsize=(10.2, 3.8))
ax = fig.add_subplot(121)
ax.loglog(true_curvature_test, predicted_curvature_test, ".",
          color=GEO_TEAL, alpha=0.35, ms=3)
curvature_limits = np.array([
    min(true_curvature_test.min(), predicted_curvature_test.min()),
    max(true_curvature_test.max(), predicted_curvature_test.max()),
])
ax.loglog(curvature_limits, curvature_limits, "--", color=GEO_RUST)
ax.set_xlabel("curvatura verdadeira"); ax.set_ylabel("curvatura prevista")
ax.set_title("transferência para um elipsoide não visto")

ax = fig.add_subplot(122, projection="3d")
relative_limit = np.quantile(np.abs(relative_curvature_error), 0.95)
scatter = ax.scatter(
    *points_ellipsoid_test.T, c=relative_curvature_error, cmap="PuOr",
    vmin=-relative_limit, vmax=relative_limit, s=5, lw=0
)
ax.set_box_aspect(test_axes); ax.set_axis_off()
ax.set_title("erro relativo sobre a superfície")
fig.colorbar(scatter, ax=ax, shrink=0.62, pad=0.02)
plt.tight_layout(); plt.show()

print(f"ridge escolhido na validação por superfície: {best_ellipsoid_ridge:.2e}")
print(f"MSE de log K no elipsoide de teste: {test_log_mse:.4f} "
      f"(baseline constante: {baseline_log_mse:.4f})")
print(f"R^2 de transferência para log K: {test_log_r2:.3f}")
print(f"erro relativo absoluto mediano: {median_relative_error:.1%}")
if test_log_r2 <= 0:
    print("Conclusão: as distâncias selecionadas não transferiram curvatura "
          "melhor que uma previsão constante para a superfície não vista.")
else:
    print("Conclusão: houve transferência parcial; o R^2 quantifica quanto da "
          "variação de log K foi recuperada na superfície não vista.")

> **Exercise 12 — double descent.** *(challenge)*
> Push the degree sweep in §8.4 to $D = 25$ (that is $3276$ monomials against $80$
> training points), keeping `np.linalg.lstsq`'s minimum-norm solution. Plot test MSE
> against $p/N$ on log axes. Do you see the error come back down past the
> interpolation threshold? Then repeat with $N_{\text{train}} = 400$ and check that
> the peak moves with $N$, not with $D$. Compare with Lecture 4.

**Minha resposta.** 

Em grau $D$, escrevemos $\binom{D+3}{3}$ monômios, mas a dimensão efetiva sobre $S^2$ é

$$p(D)=(D+1)^2.$$

Logo, o limiar de interpolação $p(D)\approx N$ deve ocorrer próximo de $D\approx\sqrt N-1$: para $N=80$, em $D=8$, quando $p=81$; para $N=400$, em $D=19$, quando $p=400$. Antes do limiar, aumentar $D$ reduz o viés mas aumenta a variância; perto de $p=N$, a inversão das pequenas direções singulares amplifica fortemente o ruído. Depois do limiar existem infinitos interpolantes, e a SVD escolhe aquele de menor norma de coeficientes. Essa escolha funciona como uma regularização implícita e pode fazer o risco diminuir novamente: o segundo ramo do double descent.

O experimento abaixo conserva exatamente a parametrização monomial redundante e a solução de norma mínima calculada por SVD. A amostra de teste independente tem 3000 pontos; isso mantém pequeno o erro de Monte Carlo sem armazenar a matriz de $20000\times3276$ que seria exigida pelo teste anterior.

In [ ]:
def explicit_double_descent_curve(
    X_train, y_train, Phi_test_max, y_test, max_degree=25
):
    """MSE da solução monomial de norma mínima para todos os graus."""
    Phi_train_max = poly_features(X_train, max_degree)
    errors = []
    numerical_ranks = []
    for degree in range(1, max_degree + 1):
        n_columns = (degree + 1) * (degree + 2) * (degree + 3) // 6
        Phi_train_degree = Phi_train_max[:, :n_columns]
        weights, _, rank, _ = np.linalg.lstsq(
            Phi_train_degree, y_train, rcond=None
        )
        prediction = Phi_test_max[:, :n_columns] @ weights
        errors.append(np.mean((prediction - y_test)**2))
        numerical_ranks.append(rank)
    return np.asarray(errors), np.asarray(numerical_ranks)


rng_ex12 = np.random.default_rng(SEED + 16)
max_double_descent_degree = 25
double_descent_degrees = np.arange(1, max_double_descent_degree + 1)
double_descent_parameters = (double_descent_degrees + 1)**2

X_double_descent_pool = sample_sphere(400, 3, rng_ex12)
y_double_descent_pool = (
    f_star(X_double_descent_pool) + NOISE * rng_ex12.normal(size=400)
)
X_double_descent_test = sample_sphere(3000, 3, rng_ex12)
y_double_descent_test = f_star(X_double_descent_test)
Phi_double_descent_test_max = poly_features(
    X_double_descent_test, max_double_descent_degree
)

double_descent_results = {}
for n_train in (80, 400):
    errors, ranks = explicit_double_descent_curve(
        X_double_descent_pool[:n_train],
        y_double_descent_pool[:n_train],
        Phi_double_descent_test_max,
        y_double_descent_test,
        max_degree=max_double_descent_degree,
    )
    double_descent_results[n_train] = {"errors": errors, "ranks": ranks}

fig, ax = plt.subplots(figsize=(6.5, 3.8))
for n_train, marker, colour in [
    (80, "o", GEO_RUST), (400, "s", GEO_TEAL)
]:
    ratios = double_descent_parameters / n_train
    errors = double_descent_results[n_train]["errors"]
    ax.loglog(ratios, errors, marker + "-", ms=4, color=colour,
              label=rf"$N={n_train}$")
    threshold_index = np.argmin(np.abs(ratios - 1))
    ax.annotate(
        rf"$D={double_descent_degrees[threshold_index]}$",
        xy=(ratios[threshold_index], errors[threshold_index]),
        xytext=(5, 7), textcoords="offset points", fontsize=7, color=colour
    )
ax.axvline(1, color=GEO_DARK, ls="--", label=r"limiar $p/N=1$")
ax.set_xlabel(r"razão parâmetros/amostras $p(D)/N$")
ax.set_ylabel("MSE de teste sem ruído")
ax.set_title("double descent: o pico acompanha p/N, não o grau")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

for n_train in (80, 400):
    errors = double_descent_results[n_train]["errors"]
    threshold_degree = int(np.ceil(np.sqrt(n_train) - 1))
    threshold_index = threshold_degree - 1
    post_threshold_minimum = errors[threshold_index:].min()
    print(
        f"N={n_train:3d}: limiar previsto D={threshold_degree}, "
        f"p={double_descent_parameters[threshold_index]}, "
        f"MSE no limiar={errors[threshold_index]:.3e}, "
        f"mínimo posterior={post_threshold_minimum:.3e}, "
        f"posto numérico={double_descent_results[n_train]['ranks'][threshold_index]}"
    )

---
## 9. What to take away

- **Sampling is a modelling decision.** "Uniform" means invariant under the
  isometry group; the naive parametrisation is wrong, and the error is invisible
  until you test a statistic whose exact distribution you know. Always have such a
  statistic.
- **Vectorise.** Every distance computation here was a Gram matrix. This is not
  only about speed; array-shaped thinking is how tensors, batches, and
  differentiable programming work from Lecture 3 on.
- **Symmetry is data structure.** $SO(3)$ acts on everything we built. Ignoring it
  wastes capacity; ignoring it *when splitting the data* invalidates the
  experiment.
- **High dimension is not more of the same.** Distances concentrate, random vectors
  are orthogonal, grids are hopeless — and the same concentration is what makes
  empirical risk approximate true risk.
- **The manifold hypothesis is testable.** Global PCA sees the ambient span; local
  PCA sees the tangent space. On a sphere embedded in $\mathbb{R}^{100}$ both give
  the right answer, and you can watch them fail as noise grows.
- **Learning is a variational problem, and the U-curve is real.** We reproduced it
  from first principles, saw the interpolation threshold sitting at $p = N$, and
  tamed it with Tikhonov regularisation — the same regularisation you already know
  from ill-posed inverse problems.

### Next

**Lecture 3** develops the convex, linear-model world where every statement above
can be proved, and **Tutorial 3** applies it to geometric classification and
regression on curves and triangulated surfaces.

### Further reading

- Vershynin, *High-Dimensional Probability* (2018), ch. 3 & 5 — concentration, rigorously; free online.
- Atkinson & Han, *Spherical Harmonics and Approximation on the Unit Sphere* (2012).
- Marques, Bobenko et al. on spherical Delaunay triangulations, for §2.3.
- Lyons, "An elementary introduction to the Hopf fibration", *Math. Mag.* **76** (2003) — the source of the pictures in §5.
- Bronstein, Bruna, Cohen & Veličković, *Geometric Deep Learning* (2021), ch. 3 — symmetry as a design principle; the destination of Exercise 10.